In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate the distribution shape, tail risk, and volatility behavior of a single asset.
#Original Risk Analysis blocks included here: 6-10.

In [ ]:
import logging
import warnings
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.io as pio
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()

from Quantapp.data import yf as qa_yf

from Quantapp.visualization import (
    BarChartPlotter,
    Plotter,
    build_time_range_buttons,)
from Quantapp.visualization.views.single_asset_profile.pricing.distribution import (
    plot_distribution_shape_zscores_view,
    plot_fixed_payout_strategy_backtest_view,
    plot_trade_range_history_profile,
    plot_trade_range_probability_cone,
    plot_trade_range_breach_average_view,
    plot_trade_range_breach_excess_view,
    plot_trade_range_stack_view,
    plot_volatility_model_comparison_view,
)
from Quantapp.analytics import (
    Helper,
    SeriesTransforms,)
from Quantapp.data import (
    MacroDataClient,
    align_series_to_common_index,
    load_benchmark_data,
    normalize_benchmark_tickers,)

# Notebook-local risk distribution analytics. Kept here because this workflow is Distribution-specific.
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

from Quantapp.analytics import compute
from Quantapp.analytics.series_utils import (
    calculate_historical_var_metrics,
    calculate_textbook_rolling_max_drawdown,
    calculate_zscore,
    coerce_close_series,
    gini_coefficient,
)


class RiskDistributionAnalytics:
    """Prepare rolling drawdown/skew/kurtosis/gini metric sets for visualization."""

    @staticmethod
    def _coerce_ohlc_frame(data) -> pd.DataFrame:
        if not isinstance(data, pd.DataFrame):
            raise TypeError("price_frame must be a pandas DataFrame with 'Open' and 'Close' columns.")

        required_columns = {"Open", "Close"}
        missing_columns = sorted(required_columns.difference(data.columns))
        if missing_columns:
            raise ValueError(
                f"price_frame is missing required columns: {missing_columns}"
            )

        frame = data.loc[:, ["Open", "Close"]].dropna().sort_index()
        if frame.empty:
            raise ValueError("price_frame is empty after dropping NaN values.")
        return frame

    @staticmethod
    def _normalize_windows(windows):
        if windows is None:
            raise ValueError("windows must be provided.")
        try:
            windows_iter = list(windows)
        except TypeError:
            windows_iter = [windows]

        normalized = []
        for window in windows_iter:
            try:
                w = int(window)
            except (TypeError, ValueError):
                continue
            if w > 0:
                normalized.append(w)

        normalized = list(dict.fromkeys(normalized))
        if not normalized:
            raise ValueError("No valid windows supplied.")
        return normalized

    @staticmethod
    def _select_default_window(window_options, default_window=None):
        if default_window in window_options:
            return default_window
        if 200 in window_options:
            return 200
        return max(window_options)

    @staticmethod
    def _normalize_confidence_levels(confidence_levels):
        if confidence_levels is None:
            confidence_levels = (0.95, 0.99)

        try:
            level_iter = list(confidence_levels)
        except TypeError:
            level_iter = [confidence_levels]

        normalized = []
        for level in level_iter:
            try:
                confidence = float(level)
            except (TypeError, ValueError):
                continue

            if confidence > 1:
                confidence = confidence / 100.0
            if 0 < confidence < 1:
                normalized.append(confidence)

        normalized = list(dict.fromkeys(normalized))
        if not normalized:
            raise ValueError("No valid confidence levels supplied.")
        return sorted(normalized)

    @staticmethod
    def _select_default_confidence(confidence_levels, default_confidence=None):
        if default_confidence is not None:
            try:
                confidence = float(default_confidence)
            except (TypeError, ValueError):
                confidence = None
            else:
                if confidence > 1:
                    confidence = confidence / 100.0
                if confidence in confidence_levels:
                    return confidence

        if 0.95 in confidence_levels:
            return 0.95
        return confidence_levels[0]

    @staticmethod
    def _normalize_horizon_sessions(horizon_sessions):
        try:
            horizon = int(horizon_sessions)
        except (TypeError, ValueError) as exc:
            raise ValueError("horizon_sessions must be a non-negative integer.") from exc
        if horizon < 0:
            raise ValueError("horizon_sessions must be a non-negative integer.")
        return horizon

    @staticmethod
    def _resolve_current_reference_price(frame):
        reference_price = frame.attrs.get("current_session_reference_price")
        if reference_price is None:
            current_session_date = frame.attrs.get("current_session_date")
            last_frame_date = pd.Timestamp(frame.index[-1]).normalize()
            if current_session_date is not None:
                current_session_date = pd.Timestamp(current_session_date).normalize()
                if current_session_date > last_frame_date:
                    reference_price = frame["Close"].iloc[-1]
                else:
                    reference_price = frame["Close"].shift(1).iloc[-1]
            else:
                reference_price = frame["Close"].shift(1).iloc[-1]
        if pd.isna(reference_price):
            reference_price = frame["Close"].iloc[-1]
        return float(reference_price)

    @staticmethod
    def _build_session_holding_period_frame(frame, horizon_sessions):
        horizon = RiskDistributionAnalytics._normalize_horizon_sessions(horizon_sessions)
        close_series = frame["Close"].astype(float)
        if horizon == 0:
            reference_series = frame["Open"].astype(float)
            exit_close = close_series
        else:
            reference_series = close_series.shift(1)
            exit_close = close_series.shift(-(horizon - 1))
        holding_frame = pd.DataFrame(
            {
                "session_open": reference_series,
                "session_close": exit_close,
                "session_return": exit_close.div(reference_series).sub(1.0),
            }
        ).dropna()
        if holding_frame.empty:
            raise ValueError(
                f"price_frame does not contain enough completed sessions for a {horizon}-session horizon."
            )
        return holding_frame

    def build_risk_distribution_context(self, close_series, windows, default_window=None):
        """
        Build drawdown/skew/kurtosis/gini rolling metrics for each window.

        Returns
        -------
        dict
            {
                "close_series": pd.Series,
                "daily_returns": pd.Series,
                "windows": list[int],
                "default_window": int,
                "metrics_by_window": dict[int, dict[str, pd.Series]],
            }
        """
        close = coerce_close_series(close_series)
        window_options = self._normalize_windows(windows)
        default_window = self._select_default_window(window_options, default_window=default_window)

        daily_returns = close.pct_change().dropna()

        def return_quantile(level):
            def quantile_metric(series):
                return series.quantile(level)

            return quantile_metric

        metrics_by_window = {}
        for window in window_options:
            max_drawdown_series = calculate_textbook_rolling_max_drawdown(close, window=window).dropna()
            rolling_return_q10 = compute.rolling(daily_returns, metric=return_quantile(0.10), window=window).dropna()
            rolling_return_q25 = compute.rolling(daily_returns, metric=return_quantile(0.25), window=window).dropna()
            rolling_return_median = compute.rolling(daily_returns, metric=pd.Series.median, window=window).dropna()
            rolling_return_q75 = compute.rolling(daily_returns, metric=return_quantile(0.75), window=window).dropna()
            rolling_return_q90 = compute.rolling(daily_returns, metric=return_quantile(0.90), window=window).dropna()
            rolling_skew = compute.rolling(
                daily_returns,
                metric=lambda series: skew(series, bias=False),
                window=window,
            ).dropna()
            rolling_kurtosis = compute.rolling(
                daily_returns,
                metric=lambda series: kurtosis(series, fisher=True, bias=False),
                window=window,
            ).dropna()
            rolling_gini = compute.rolling(daily_returns, metric=gini_coefficient, window=window).dropna()

            metrics_by_window[window] = {
                "daily_returns": daily_returns.copy(),
                "return_q10": rolling_return_q10,
                "return_q25": rolling_return_q25,
                "return_median": rolling_return_median,
                "return_q75": rolling_return_q75,
                "return_q90": rolling_return_q90,
                "max_drawdown": max_drawdown_series,
                "skew_z": calculate_zscore(rolling_skew),
                "kurtosis_z": calculate_zscore(rolling_kurtosis),
                "gini_z": calculate_zscore(rolling_gini),
            }

        return {
            "close_series": close,
            "daily_returns": daily_returns,
            "windows": window_options,
            "default_window": default_window,
            "metrics_by_window": metrics_by_window,
        }

    def build_value_at_risk_context(
        self,
        close_series,
        windows,
        confidence_levels=(0.95, 0.99),
        default_window=None,
        default_confidence=None,
        position_value=None,
    ):
        """
        Build rolling historical VaR / CVaR (Expected Shortfall) metrics for each window and confidence level.

        Returns
        -------
        dict
            {
                "close_series": pd.Series,
                "daily_returns": pd.Series,
                "windows": list[int],
                "default_window": int,
                "confidence_levels": list[float],
                "default_confidence": float,
                "metrics_by_window": dict[int, dict[float, dict[str, pd.Series]]],
                "summary_table": pd.DataFrame,
                "position_value": float | None,
            }
        """
        close = coerce_close_series(close_series)
        window_options = self._normalize_windows(windows)
        default_window = self._select_default_window(window_options, default_window=default_window)
        confidence_levels = self._normalize_confidence_levels(confidence_levels)
        default_confidence = self._select_default_confidence(
            confidence_levels,
            default_confidence=default_confidence,
        )

        daily_returns = close.pct_change(fill_method=None).dropna()
        metrics_by_window = {}
        summary_rows = []

        for window in window_options:
            metrics_by_confidence = {}
            for confidence in confidence_levels:
                alpha = 1 - confidence
                metric_set = calculate_historical_var_metrics(daily_returns, window=window, alpha=alpha)

                if position_value is not None:
                    metric_set["var_dollar"] = metric_set["var"] * float(position_value)
                    metric_set["expected_shortfall_dollar"] = (
                        metric_set["expected_shortfall"] * float(position_value)
                    )

                metrics_by_confidence[confidence] = metric_set

                var_series = metric_set["var"].dropna()
                es_series = metric_set["expected_shortfall"].dropna()
                breach_series = metric_set["breaches"].dropna()
                rolling_breach_rate = metric_set["rolling_breach_rate"].dropna()

                summary_row = {
                    "VaR Lookback Window": window,
                    "Confidence": f"{confidence:.0%}",
                    "Latest VaR": var_series.iloc[-1] if not var_series.empty else np.nan,
                    "Latest CVaR": es_series.iloc[-1] if not es_series.empty else np.nan,
                    "Observed Breach Rate": breach_series.mean() if not breach_series.empty else np.nan,
                    "Expected Breach Rate": alpha,
                    "Latest Rolling Breach Rate": (
                        rolling_breach_rate.iloc[-1] if not rolling_breach_rate.empty else np.nan
                    ),
                }

                if position_value is not None:
                    var_dollar = metric_set["var_dollar"].dropna()
                    es_dollar = metric_set["expected_shortfall_dollar"].dropna()
                    summary_row["Latest VaR Dollar"] = var_dollar.iloc[-1] if not var_dollar.empty else np.nan
                    summary_row["Latest CVaR Dollar"] = es_dollar.iloc[-1] if not es_dollar.empty else np.nan

                summary_rows.append(summary_row)

            metrics_by_window[window] = metrics_by_confidence

        summary_table = pd.DataFrame(summary_rows)
        if not summary_table.empty:
            summary_table = summary_table.sort_values(["VaR Lookback Window", "Confidence"]).reset_index(drop=True)

        return {
            "close_series": close,
            "daily_returns": daily_returns,
            "windows": window_options,
            "default_window": default_window,
            "confidence_levels": confidence_levels,
            "default_confidence": default_confidence,
            "metrics_by_window": metrics_by_window,
            "summary_table": summary_table,
            "position_value": position_value,
        }

    def build_session_probability_cone_context(
        self,
        price_frame,
        window=200,
        interval_confidence_levels=(0.50, 0.80, 0.90, 0.95),
        var_confidence_levels=(0.95, 0.99),
        anchor_price=None,
        latest_price=None,
        session_date=None,
    ):
        """
        Build an open-anchored probability cone for the latest session using trailing
        open-to-close returns from completed sessions.

        Returns
        -------
        dict
            {
                "session_date": pd.Timestamp,
                "window": int,
                "effective_window": int,
                "anchor_price": float,
                "latest_price": float,
                "sample_returns": pd.Series,
                "interval_confidence_levels": list[float],
                "var_confidence_levels": list[float],
                "intervals": dict[float, dict[str, float]],
                "var_levels": dict[float, dict[str, float]],
                "median_return": float,
                "median_price": float,
                "summary_table": pd.DataFrame,
            }
        """
        frame = self._coerce_ohlc_frame(price_frame)
        try:
            window = int(window)
        except (TypeError, ValueError) as exc:
            raise ValueError("window must be a positive integer.") from exc
        if window <= 0:
            raise ValueError("window must be a positive integer.")

        interval_confidence_levels = self._normalize_confidence_levels(interval_confidence_levels)
        var_confidence_levels = self._normalize_confidence_levels(var_confidence_levels)

        session_returns = frame["Close"].div(frame["Open"]).sub(1.0).dropna()
        if len(session_returns) < 2:
            raise ValueError(
                "At least two sessions with valid open and close prices are required "
                "to build a current-session probability cone."
            )

        historical_returns = session_returns.iloc[:-1].dropna()
        if historical_returns.empty:
            raise ValueError("No completed session returns are available for the cone sample.")

        effective_window = min(window, len(historical_returns))
        sample_returns = historical_returns.tail(effective_window)
        session_date = pd.Timestamp(
            frame.attrs.get("current_session_date", frame.index[-1])
            if session_date is None
            else session_date
        )
        latest_price = float(
            frame.attrs.get("current_session_latest_price", frame["Close"].iloc[-1])
            if latest_price is None
            else latest_price
        )

        if anchor_price is None:
            anchor_price = float(
                frame.attrs.get("current_session_anchor_price", frame["Open"].iloc[-1])
            )
        else:
            anchor_price = float(anchor_price)

        if not np.isfinite(anchor_price) or anchor_price <= 0:
            raise ValueError("anchor_price must be a positive finite value.")

        median_return = float(sample_returns.median())
        median_price = anchor_price * (1.0 + median_return)

        interval_map = {}
        summary_rows = []
        for confidence in interval_confidence_levels:
            tail_probability = (1.0 - confidence) / 2.0
            lower_return = float(sample_returns.quantile(tail_probability))
            upper_return = float(sample_returns.quantile(1.0 - tail_probability))
            lower_price = anchor_price * (1.0 + lower_return)
            upper_price = anchor_price * (1.0 + upper_return)
            interval_map[confidence] = {
                "lower_return": lower_return,
                "upper_return": upper_return,
                "lower_price": lower_price,
                "upper_price": upper_price,
            }
            summary_rows.append(
                {
                    "Metric": f"{confidence:.0%} Central Range",
                    "Lower Return": lower_return,
                    "Upper Return": upper_return,
                    "Lower Price": lower_price,
                    "Upper Price": upper_price,
                }
            )

        var_level_map = {}
        for confidence in var_confidence_levels:
            alpha = 1.0 - confidence
            metric_set = calculate_historical_var_metrics(
                sample_returns,
                window=effective_window,
                alpha=alpha,
            )
            var_series = metric_set["var"].dropna()
            expected_shortfall_series = metric_set["expected_shortfall"].dropna()

            var_loss = float(var_series.iloc[-1]) if not var_series.empty else np.nan
            expected_shortfall_loss = (
                float(expected_shortfall_series.iloc[-1])
                if not expected_shortfall_series.empty
                else np.nan
            )
            if pd.isna(var_loss):
                var_loss = float(max(-float(sample_returns.quantile(alpha)), 0.0))
            if pd.isna(expected_shortfall_loss):
                tail_values = sample_returns[sample_returns <= sample_returns.quantile(alpha)]
                tail_mean = float(tail_values.mean()) if not tail_values.empty else -var_loss
                expected_shortfall_loss = float(max(-tail_mean, var_loss))

            var_return = -var_loss
            expected_shortfall_return = -expected_shortfall_loss
            var_price = anchor_price * (1.0 + var_return)
            expected_shortfall_price = anchor_price * (1.0 + expected_shortfall_return)

            var_level_map[confidence] = {
                "var_loss": var_loss,
                "var_return": var_return,
                "var_price": var_price,
                "expected_shortfall_loss": expected_shortfall_loss,
                "expected_shortfall_return": expected_shortfall_return,
                "expected_shortfall_price": expected_shortfall_price,
            }
            summary_rows.extend(
                [
                    {
                        "Metric": f"{confidence:.0%} VaR Floor",
                        "Lower Return": var_return,
                        "Upper Return": np.nan,
                        "Lower Price": var_price,
                        "Upper Price": np.nan,
                    },
                    {
                        "Metric": f"{confidence:.0%} CVaR Floor",
                        "Lower Return": expected_shortfall_return,
                        "Upper Return": np.nan,
                        "Lower Price": expected_shortfall_price,
                        "Upper Price": np.nan,
                    },
                ]
            )

        summary_rows.insert(
            0,
            {
                "Metric": "Session Open",
                "Lower Return": 0.0,
                "Upper Return": 0.0,
                "Lower Price": anchor_price,
                "Upper Price": anchor_price,
            },
        )
        summary_rows.insert(
            1,
            {
                "Metric": "Latest Session Price",
                "Lower Return": (latest_price / anchor_price) - 1.0 if anchor_price else np.nan,
                "Upper Return": np.nan,
                "Lower Price": latest_price,
                "Upper Price": np.nan,
            },
        )

        summary_table = pd.DataFrame(summary_rows)

        return {
            "session_date": session_date,
            "window": window,
            "effective_window": effective_window,
            "anchor_price": anchor_price,
            "latest_price": latest_price,
            "sample_returns": sample_returns,
            "interval_confidence_levels": interval_confidence_levels,
            "var_confidence_levels": var_confidence_levels,
            "intervals": interval_map,
            "var_levels": var_level_map,
            "median_return": median_return,
            "median_price": median_price,
            "summary_table": summary_table,
        }

    def build_trade_range_probability_context(
        self,
        price_frame,
        window=200,
        horizon_sessions=1,
        interval_confidence_levels=(0.95, 0.99),
        tail_confidence_levels=(0.95, 0.99),
        anchor_price=None,
        latest_price=None,
        session_date=None,
    ):
        """
        Build a two-sided close-based session probability context for long and short
        decision support using trailing completed holding-period returns.
        """
        frame = self._coerce_ohlc_frame(price_frame)
        try:
            window = int(window)
        except (TypeError, ValueError) as exc:
            raise ValueError("window must be a positive integer.") from exc
        if window <= 0:
            raise ValueError("window must be a positive integer.")
        horizon_sessions = self._normalize_horizon_sessions(horizon_sessions)

        interval_confidence_levels = self._normalize_confidence_levels(interval_confidence_levels)
        tail_confidence_levels = self._normalize_confidence_levels(tail_confidence_levels)

        holding_frame = self._build_session_holding_period_frame(frame, horizon_sessions)
        session_returns = holding_frame["session_return"]
        if len(session_returns) < 2:
            raise ValueError(
                "At least two completed holding-period returns are required "
                "to build a current-session probability range."
            )

        historical_returns = session_returns.iloc[:-1].dropna()
        if historical_returns.empty:
            raise ValueError("No completed holding-period returns are available for the trade range sample.")

        effective_window = min(window, len(historical_returns))
        sample_returns = historical_returns.tail(effective_window)
        session_date = pd.Timestamp(
            frame.attrs.get("current_session_date", frame.index[-1])
            if session_date is None
            else session_date
        )
        latest_price = float(
            frame.attrs.get("current_session_latest_price", frame["Close"].iloc[-1])
            if latest_price is None
            else latest_price
        )

        if anchor_price is None:
            if horizon_sessions == 0:
                anchor_price = frame.attrs.get("current_session_anchor_price", frame["Open"].iloc[-1])
            else:
                anchor_price = self._resolve_current_reference_price(frame)
        else:
            anchor_price = float(anchor_price)

        if not np.isfinite(anchor_price) or anchor_price <= 0:
            raise ValueError("anchor_price must be a positive finite value.")

        median_return = float(sample_returns.median())
        median_price = anchor_price * (1.0 + median_return)

        interval_map = {}
        range_rows = []
        for confidence in interval_confidence_levels:
            tail_probability = (1.0 - confidence) / 2.0
            lower_return = float(sample_returns.quantile(tail_probability))
            upper_return = float(sample_returns.quantile(1.0 - tail_probability))
            lower_price = anchor_price * (1.0 + lower_return)
            upper_price = anchor_price * (1.0 + upper_return)
            interval_map[confidence] = {
                "lower_return": lower_return,
                "upper_return": upper_return,
                "lower_price": lower_price,
                "upper_price": upper_price,
            }
            range_rows.append(
                {
                    "Confidence": f"{confidence:.0%}",
                    "Lower Return": lower_return,
                    "Upper Return": upper_return,
                    "Lower Close": lower_price,
                    "Upper Close": upper_price,
                }
            )

        long_tail_map = {}
        short_tail_map = {}
        tail_rows = []
        for confidence in tail_confidence_levels:
            alpha = 1.0 - confidence

            lower_cutoff = float(sample_returns.quantile(alpha))
            upper_cutoff = float(sample_returns.quantile(1.0 - alpha))

            lower_tail_values = sample_returns[sample_returns <= lower_cutoff]
            upper_tail_values = sample_returns[sample_returns >= upper_cutoff]

            long_cvar_return = (
                float(lower_tail_values.mean())
                if not lower_tail_values.empty
                else lower_cutoff
            )
            short_cvar_return = (
                float(upper_tail_values.mean())
                if not upper_tail_values.empty
                else upper_cutoff
            )

            long_tail_map[confidence] = {
                "var_return": lower_cutoff,
                "var_price": anchor_price * (1.0 + lower_cutoff),
                "expected_shortfall_return": long_cvar_return,
                "expected_shortfall_price": anchor_price * (1.0 + long_cvar_return),
            }
            short_tail_map[confidence] = {
                "var_return": upper_cutoff,
                "var_price": anchor_price * (1.0 + upper_cutoff),
                "expected_shortfall_return": short_cvar_return,
                "expected_shortfall_price": anchor_price * (1.0 + short_cvar_return),
            }

            tail_rows.append(
                {
                    "Confidence": f"{confidence:.0%}",
                    "Long VaR Floor": long_tail_map[confidence]["var_price"],
                    "Long CVaR Floor": long_tail_map[confidence]["expected_shortfall_price"],
                    "Short VaR Ceiling": short_tail_map[confidence]["var_price"],
                    "Short CVaR Ceiling": short_tail_map[confidence]["expected_shortfall_price"],
                }
            )

        return {
            "session_date": session_date,
            "window": window,
            "horizon_sessions": horizon_sessions,
            "effective_window": effective_window,
            "anchor_price": anchor_price,
            "latest_price": latest_price,
            "sample_returns": sample_returns,
            "return_basis": "open_to_close" if horizon_sessions == 0 else "close_to_close",
            "interval_confidence_levels": interval_confidence_levels,
            "tail_confidence_levels": tail_confidence_levels,
            "intervals": interval_map,
            "long_tail_levels": long_tail_map,
            "short_tail_levels": short_tail_map,
            "median_return": median_return,
            "median_price": median_price,
            "range_summary_table": pd.DataFrame(range_rows),
            "tail_summary_table": pd.DataFrame(tail_rows),
        }

    def build_trade_range_history_context(
        self,
        price_frame,
        window=200,
        windows=None,
        horizon_sessions=1,
        interval_confidence_levels=(0.95, 0.99),
        tail_confidence_levels=(0.95, 0.99),
        default_window=None,
    ):
        """
        Build ex-ante historical session trade-range metrics for one or more rolling
        windows using only information available before each entry session.
        """
        frame = self._coerce_ohlc_frame(price_frame)
        window_seed = windows if windows is not None else [window]
        window_options = self._normalize_windows(window_seed)
        default_window = self._select_default_window(
            window_options,
            default_window=default_window,
        )
        horizon_sessions = self._normalize_horizon_sessions(horizon_sessions)
        interval_confidence_levels = self._normalize_confidence_levels(interval_confidence_levels)
        tail_confidence_levels = self._normalize_confidence_levels(tail_confidence_levels)

        holding_frame = self._build_session_holding_period_frame(frame, horizon_sessions)
        open_series = holding_frame["session_open"]
        close_series = holding_frame["session_close"]
        session_returns = holding_frame["session_return"]
        max_window = max(window_options)
        minimum_observations = max_window + horizon_sessions - 1
        if len(session_returns) <= minimum_observations:
            raise ValueError(
                f"Not enough completed sessions to build a {max_window}-session lookback with a "
                f"{horizon_sessions}-session horizon."
            )

        def lower_tail_mean(values, quantile_level):
            arr = np.asarray(values, dtype=float)
            arr = arr[~np.isnan(arr)]
            if arr.size == 0:
                return np.nan
            cutoff = np.quantile(arr, quantile_level)
            tail_values = arr[arr <= cutoff]
            if tail_values.size == 0:
                return cutoff
            return tail_values.mean()

        def upper_tail_mean(values, quantile_level):
            arr = np.asarray(values, dtype=float)
            arr = arr[~np.isnan(arr)]
            if arr.size == 0:
                return np.nan
            cutoff = np.quantile(arr, quantile_level)
            tail_values = arr[arr >= cutoff]
            if tail_values.size == 0:
                return cutoff
            return tail_values.mean()

        all_confidences = sorted(set(interval_confidence_levels).union(tail_confidence_levels))
        metrics_by_window = {}
        for rolling_window in window_options:
            metrics_by_confidence = {}
            for confidence in all_confidences:
                alpha = 1.0 - confidence
                interval_alpha = (1.0 - confidence) / 2.0
                shift_periods = horizon_sessions

                lower_interval_return = (
                    session_returns.rolling(rolling_window).quantile(interval_alpha).shift(shift_periods).dropna()
                )
                upper_interval_return = (
                    session_returns.rolling(rolling_window).quantile(1.0 - interval_alpha).shift(shift_periods).dropna()
                )
                lower_var_return = session_returns.rolling(rolling_window).quantile(alpha).shift(shift_periods).dropna()
                upper_var_return = session_returns.rolling(rolling_window).quantile(1.0 - alpha).shift(shift_periods).dropna()
                lower_expected_shortfall_return = (
                    session_returns
                    .rolling(rolling_window)
                    .apply(lambda values, q=alpha: lower_tail_mean(values, q), raw=True)
                    .shift(shift_periods)
                    .dropna()
                )
                upper_expected_shortfall_return = (
                    session_returns
                    .rolling(rolling_window)
                    .apply(lambda values, q=1.0 - alpha: upper_tail_mean(values, q), raw=True)
                    .shift(shift_periods)
                    .dropna()
                )

                aligned_returns = session_returns.reindex(lower_var_return.index).dropna()
                lower_var_return = lower_var_return.reindex(aligned_returns.index)
                upper_var_return = upper_var_return.reindex(aligned_returns.index)
                lower_breaches = aligned_returns.lt(lower_var_return).astype(float)
                upper_breaches = aligned_returns.gt(upper_var_return).astype(float)
                either_side_breaches = lower_breaches.add(upper_breaches, fill_value=0.0).clip(upper=1.0)
                lower_rolling_breach_rate = lower_breaches.rolling(rolling_window).mean().dropna()
                upper_rolling_breach_rate = upper_breaches.rolling(rolling_window).mean().dropna()
                either_side_rolling_breach_rate = (
                    either_side_breaches.rolling(rolling_window).mean().dropna()
                )

                lower_expected_breach_rate = pd.Series(
                    data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
                    index=lower_rolling_breach_rate.index,
                )
                upper_expected_breach_rate = pd.Series(
                    data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
                    index=upper_rolling_breach_rate.index,
                )
                either_side_expected_breach_rate = pd.Series(
                    data=np.full(
                        len(either_side_rolling_breach_rate.index),
                        min(1.0, 2.0 * alpha),
                        dtype=float,
                    ),
                    index=either_side_rolling_breach_rate.index,
                )

                open_for_interval = open_series.reindex(lower_interval_return.index)
                open_for_var = open_series.reindex(lower_var_return.index)
                open_for_es = open_series.reindex(lower_expected_shortfall_return.index)

                metrics_by_confidence[confidence] = {
                    "session_returns": session_returns,
                    "session_open": open_series,
                    "session_close": close_series,
                    "lower_interval_return": lower_interval_return,
                    "upper_interval_return": upper_interval_return,
                    "lower_interval_price": open_for_interval.mul(1.0 + lower_interval_return),
                    "upper_interval_price": open_for_interval.mul(1.0 + upper_interval_return),
                    "lower_var_return": lower_var_return,
                    "upper_var_return": upper_var_return,
                    "lower_var_price": open_for_var.mul(1.0 + lower_var_return),
                    "upper_var_price": open_for_var.mul(1.0 + upper_var_return),
                    "lower_expected_shortfall_return": lower_expected_shortfall_return,
                    "upper_expected_shortfall_return": upper_expected_shortfall_return,
                    "lower_expected_shortfall_price": open_for_es.mul(1.0 + lower_expected_shortfall_return),
                    "upper_expected_shortfall_price": open_for_es.mul(1.0 + upper_expected_shortfall_return),
                    "lower_breaches": lower_breaches,
                    "upper_breaches": upper_breaches,
                    "either_side_breaches": either_side_breaches,
                    "lower_rolling_breach_rate": lower_rolling_breach_rate,
                    "upper_rolling_breach_rate": upper_rolling_breach_rate,
                    "either_side_rolling_breach_rate": either_side_rolling_breach_rate,
                    "lower_expected_breach_rate": lower_expected_breach_rate,
                    "upper_expected_breach_rate": upper_expected_breach_rate,
                    "either_side_expected_breach_rate": either_side_expected_breach_rate,
                }

            metrics_by_window[rolling_window] = metrics_by_confidence

        return {
            "window": default_window,
            "windows": window_options,
            "default_window": default_window,
            "horizon_sessions": horizon_sessions,
            "interval_confidence_levels": interval_confidence_levels,
            "tail_confidence_levels": tail_confidence_levels,
            "metrics_by_confidence": metrics_by_window[default_window],
            "metrics_by_window": metrics_by_window,
            "return_basis": "open_to_close" if horizon_sessions == 0 else "close_to_close",
            "session_returns": session_returns,
            "session_open": open_series,
            "session_close": close_series,
        }
warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
series_transforms = SeriesTransforms()
risk_distribution_analytics = RiskDistributionAnalytics()


In [ ]:
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
barChartPlotter = BarChartPlotter()

PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    fig.show(config=merged_config)

In [ ]:
# Block 3: set notebook parameters

TIMEFRAME_PROFILES = {
    "swing": {"short": 3, "mid": 9, "long": 21},
    "position": {"short": 21, "mid": 50, "long": 200},
    "structural": {"short": 200, "mid": 500, "long": 1000},
}


def resolve_time_frame_map(strategy: str) -> dict[str, int]:
    normalized_strategy = str(strategy).strip().lower()
    if normalized_strategy not in TIMEFRAME_PROFILES:
        raise ValueError(
            f"Invalid trading_strategy '{strategy}'. "
            f"Expected one of: {list(TIMEFRAME_PROFILES.keys())}"
        )
    return dict(TIMEFRAME_PROFILES[normalized_strategy])

single_asset_params = {
    "ticker_str": "TLT",
    "interval": "1d",
    "period": "20y",
    "risk_free_ticker": "^IRX",
    "benchmark_tickers": ["SPY"],
    "trading_strategy": "position",
    "length_of_plots": 20,
    "var_position_value": None,
}

ticker_str = single_asset_params["ticker_str"]
interval = single_asset_params["interval"]
period = single_asset_params["period"]
risk_free_ticker = single_asset_params["risk_free_ticker"]
benchmark_tickers = list(single_asset_params["benchmark_tickers"])
trading_strategy = single_asset_params["trading_strategy"]
length_of_plots = single_asset_params["length_of_plots"]
var_position_value = single_asset_params["var_position_value"]

single_asset_params

In [ ]:
# Block 4: organize structural, position, swing timeframe lists

strategy = str(trading_strategy).strip().lower()
time_frame_map = resolve_time_frame_map(strategy)
time_frame_short = time_frame_map["short"]
time_frame_mid = time_frame_map["mid"]
time_frame_long = time_frame_map["long"]

return_frequencies = ('monthly', 'weekly', 'daily')

In [ ]:
# Block 5: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = qa_yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = qa_yf.Ticker('^VIX').history(period=period, interval=interval)
risk_free_proxy = qa_yf.Ticker(risk_free_ticker).history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
ticker_intraday = None
if helper.is_futures_ticker(ticker_str):
    try:
        ticker_intraday = qa_yf.Ticker(ticker_str).history(period='60d', interval='30m')
    except Exception:
        ticker_intraday = None
ticker_trade_range_source = helper.build_equity_like_trade_range_source(
    ticker_str,
    ticker,
    intraday_frame=ticker_intraday,
)
vix = helper.simplify_datetime_index(vix)
risk_free_proxy = helper.simplify_datetime_index(risk_free_proxy)
if risk_free_proxy.empty or 'Close' not in risk_free_proxy:
    raise ValueError(f"No risk-free history available for {risk_free_ticker}.")
risk_free_annual_yield = risk_free_proxy['Close'].dropna().sort_index().div(100)
risk_free_daily_rate = ((1 + risk_free_annual_yield) ** (1 / 252) - 1).shift(1)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str, include_asset=True)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)
risk_free_daily_rate = risk_free_daily_rate.reindex(ticker.index).ffill()

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']

In [ ]:
# Block 6: analyze daily returns, skew, kurtosis, and Gini z-scores

distribution_window_options = [21, 50, 200]

distribution_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=distribution_window_options,
    default_window=200 if 200 in distribution_window_options else max(distribution_window_options),
)

fig = plot_distribution_shape_zscores_view(
    metrics_by_window=distribution_context['metrics_by_window'],
    window_options=distribution_context['windows'],
    default_window=distribution_context['default_window'],
    ticker_label=ticker_str,
    include_return_panel=False,
)
show_plotly_figure(fig)

In [ ]:
# Block 7: compare close-to-close trade-range methods with Dash selectors



trade_range_price_frame = globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].dropna().copy()
trade_range_window_candidates = [21, 50, 200]
trade_range_max_supported_window = max(1, len(trade_range_price_frame) - 1)
trade_range_window_options = [window for window in trade_range_window_candidates if window <= trade_range_max_supported_window]
if not trade_range_window_options:
    raise ValueError(
        f'Trade-range analysis needs at least 21 completed sessions. '
        f'Only {trade_range_max_supported_window} are available for {ticker_str} '
        f'using {trade_range_price_frame.attrs.get("session_mode", "the current session mode")}.'
    )
trade_range_default_window = 50 if 50 in trade_range_window_options else max(trade_range_window_options)
trade_range_default_horizon = 0
trade_range_interval_levels = [0.85, 0.95, 0.99]
trade_range_tail_levels = [0.85, 0.95, 0.99]
trade_range_confidence_options = sorted(set(trade_range_interval_levels).union(trade_range_tail_levels))
trade_range_default_confidence = 0.95 if 0.95 in trade_range_confidence_options else trade_range_confidence_options[0]

trade_range_history_context = risk_distribution_analytics.build_trade_range_history_context(
    price_frame=trade_range_price_frame,
    windows=trade_range_window_options,
    horizon_sessions=trade_range_default_horizon,
    interval_confidence_levels=trade_range_interval_levels,
    tail_confidence_levels=trade_range_tail_levels,
    default_window=trade_range_default_window,
)
trade_range_history_contexts_by_horizon = {
    int(trade_range_default_horizon): trade_range_history_context,
}

trade_range_history_fig = plot_trade_range_history_profile(
    history_context=trade_range_history_context,
    ticker_label=ticker_str,
)
import copy as _copy

trade_range_cone_contexts_by_key = {
    (int(trade_range_default_window), int(trade_range_default_horizon)): (
        risk_distribution_analytics.build_trade_range_probability_context(
            price_frame=trade_range_price_frame,
            window=int(trade_range_default_window),
            horizon_sessions=int(trade_range_default_horizon),
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
        )
    )
}

trade_range_window = trade_range_history_context['default_window']
trade_range_horizon = int(trade_range_default_horizon)
trade_range_context = trade_range_cone_contexts_by_key[(trade_range_window, trade_range_horizon)]

range_summary = trade_range_context['range_summary_table'].copy()
tail_summary = trade_range_context['tail_summary_table'].copy()

range_summary = range_summary.rename(columns={
    'Lower Close': 'Lower Exit Price',
    'Upper Close': 'Upper Exit Price',
})
tail_summary = tail_summary.rename(columns={
    'Confidence': 'Tail Confidence',
})

if not range_summary.empty:
    for column in ('Lower Return', 'Upper Return'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Lower Exit Price', 'Upper Exit Price'):
        if column in range_summary.columns:
            range_summary[column] = range_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )

if not tail_summary.empty:
    for column in ('Long VaR Floor', 'Long CVaR Floor', 'Short VaR Ceiling', 'Short CVaR Ceiling'):
        if column in tail_summary.columns:
            tail_summary[column] = tail_summary[column].map(
                lambda value: f'${value:,.2f}' if pd.notna(value) else value
            )

# Build the optional GARCH(1,1)-normal variant so it can share the same selector.
for _garch_output_name in (
    'garch_trade_history_context',
    'formatted_garch_model_summary',
    'garch_range_summary',
    'garch_tail_summary',
    'garch_trade_combined_fig',
    'garch_trade_range_fig',
    'garch_cone_components_by_key',
    'garch_trade_history_contexts_by_key',
    'build_garch_trade_range_history_context',
    'garch_cone_components_by_window',
    'garch_default_components',
):
    globals().pop(_garch_output_name, None)

garch_option_error = None
try:


    from scipy.stats import norm

    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != 'arch':
            raise
        raise ImportError(
            "The GARCH option requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc

    trade_range_garch_window = trade_range_default_window if 'trade_range_default_window' in globals() else 50
    trade_range_garch_window_options = trade_range_window_options if 'trade_range_window_options' in globals() else [trade_range_garch_window]
    trade_range_garch_default_horizon = 1
    trade_range_garch_interval_levels = trade_range_interval_levels if 'trade_range_interval_levels' in globals() else [0.95, 0.99]
    trade_range_garch_tail_levels = trade_range_tail_levels if 'trade_range_tail_levels' in globals() else [0.95, 0.99]

    # Fit the model to completed close-to-close returns so the forecast stays aligned with the simplified trade-range view.
    garch_trade_frame = trade_range_price_frame.dropna().copy()
    garch_session_returns = garch_trade_frame['Close'].pct_change().dropna()
    if len(garch_session_returns) < 3:
        raise ValueError(
            'The GARCH option needs at least three sessions with valid close prices to fit a GARCH trade-range variant.'
        )

    garch_completed_frame = garch_trade_frame.iloc[:-1].copy()
    garch_completed_returns = garch_trade_frame['Close'].pct_change().iloc[:-1].dropna()
    if len(garch_completed_returns) < 50:
        raise ValueError(
            f'The GARCH option needs at least 50 completed sessions to fit a stable GARCH trade-range variant. Only {len(garch_completed_returns)} are available.'
        )

    # Historical diagnostics use a filtered full-sample GARCH fit across completed sessions.
    garch_history_model = arch_model(
        garch_completed_returns.mul(100.0),
        mean='Constant',
        vol='GARCH',
        p=1,
        q=1,
        dist='normal',
        rescale=False,
    )
    garch_history_fit = garch_history_model.fit(disp='off')
    garch_history_mean_return = float(garch_history_fit.params.get('mu', 0.0) / 100.0)
    garch_history_sigma_series = pd.Series(
        garch_history_fit.conditional_volatility / 100.0,
        index=garch_completed_returns.index,
    )
    garch_history_open = garch_trade_frame['Close'].shift(1).astype(float).reindex(garch_completed_returns.index)
    garch_history_close = garch_completed_frame['Close'].astype(float).reindex(garch_completed_returns.index)
    garch_current_history_forecast = garch_history_fit.forecast(horizon=1, reindex=False)
    garch_current_history_mean_return = float(garch_current_history_forecast.mean.iloc[-1, 0] / 100.0)
    garch_current_history_sigma_return = float(np.sqrt(garch_current_history_forecast.variance.iloc[-1, 0]) / 100.0)
    if not np.isfinite(garch_current_history_sigma_return) or garch_current_history_sigma_return <= 0:
        raise ValueError(
            'The GARCH option could not produce a positive current-session GARCH volatility forecast for the historical diagnostics.'
        )
    garch_current_history_date = garch_trade_frame.index[-1]
    garch_current_history_return = float((garch_trade_frame.attrs.get('current_session_latest_price', garch_trade_frame['Close'].iloc[-1]) / risk_distribution_analytics._resolve_current_reference_price(garch_trade_frame)) - 1.0)
    garch_history_metrics_by_confidence = {}
    for confidence in sorted(set(trade_range_garch_interval_levels).union(trade_range_garch_tail_levels)):
        alpha = 1.0 - confidence
        interval_alpha = (1.0 - confidence) / 2.0
        interval_z = float(norm.ppf(interval_alpha))
        z_alpha = float(norm.ppf(alpha))
        pdf_alpha = float(norm.pdf(z_alpha))

        lower_interval_return = garch_history_mean_return + (garch_history_sigma_series * interval_z)
        upper_interval_return = garch_history_mean_return - (garch_history_sigma_series * interval_z)
        lower_var_return = garch_history_mean_return + (garch_history_sigma_series * z_alpha)
        upper_var_return = garch_history_mean_return - (garch_history_sigma_series * z_alpha)
        lower_expected_shortfall_return = garch_history_mean_return - (garch_history_sigma_series * (pdf_alpha / alpha))
        upper_expected_shortfall_return = garch_history_mean_return + (garch_history_sigma_series * (pdf_alpha / alpha))

        current_lower_var_return = garch_current_history_mean_return + (garch_current_history_sigma_return * z_alpha)
        current_upper_var_return = garch_current_history_mean_return - (garch_current_history_sigma_return * z_alpha)
        current_lower_expected_shortfall_return = (
            garch_current_history_mean_return - (garch_current_history_sigma_return * (pdf_alpha / alpha))
        )
        current_upper_expected_shortfall_return = (
            garch_current_history_mean_return + (garch_current_history_sigma_return * (pdf_alpha / alpha))
        )

        lower_var_return_with_current = pd.concat([
            lower_var_return,
            pd.Series([current_lower_var_return], index=[garch_current_history_date]),
        ]).sort_index()
        upper_var_return_with_current = pd.concat([
            upper_var_return,
            pd.Series([current_upper_var_return], index=[garch_current_history_date]),
        ]).sort_index()
        lower_expected_shortfall_return_with_current = pd.concat([
            lower_expected_shortfall_return,
            pd.Series([current_lower_expected_shortfall_return], index=[garch_current_history_date]),
        ]).sort_index()
        upper_expected_shortfall_return_with_current = pd.concat([
            upper_expected_shortfall_return,
            pd.Series([current_upper_expected_shortfall_return], index=[garch_current_history_date]),
        ]).sort_index()

        lower_breaches = garch_completed_returns.lt(lower_var_return).astype(float)
        upper_breaches = garch_completed_returns.gt(upper_var_return).astype(float)
        lower_breaches_with_current = pd.concat([
            lower_breaches,
            pd.Series([float(garch_current_history_return < current_lower_var_return)], index=[garch_current_history_date]),
        ]).sort_index()
        upper_breaches_with_current = pd.concat([
            upper_breaches,
            pd.Series([float(garch_current_history_return > current_upper_var_return)], index=[garch_current_history_date]),
        ]).sort_index()
        either_side_breaches_with_current = lower_breaches_with_current.add(upper_breaches_with_current, fill_value=0.0).clip(upper=1.0)
        lower_rolling_breach_rate = lower_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()
        upper_rolling_breach_rate = upper_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()
        either_side_rolling_breach_rate = either_side_breaches_with_current.rolling(trade_range_garch_window).mean().dropna()

        lower_expected_breach_rate = pd.Series(
            data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
            index=lower_rolling_breach_rate.index,
        )
        upper_expected_breach_rate = pd.Series(
            data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
            index=upper_rolling_breach_rate.index,
        )
        either_side_expected_breach_rate = pd.Series(
            data=np.full(len(either_side_rolling_breach_rate.index), min(1.0, 2.0 * alpha), dtype=float),
            index=either_side_rolling_breach_rate.index,
        )

        open_for_interval = garch_history_open.reindex(lower_interval_return.index)
        open_for_var = garch_history_open.reindex(lower_var_return.index)
        open_for_es = garch_history_open.reindex(lower_expected_shortfall_return.index)

        garch_history_metrics_by_confidence[confidence] = {
            'session_returns': garch_completed_returns,
            'session_open': garch_history_open,
            'session_close': garch_history_close,
            'lower_interval_return': lower_interval_return,
            'upper_interval_return': upper_interval_return,
            'lower_interval_price': open_for_interval.mul(1.0 + lower_interval_return),
            'upper_interval_price': open_for_interval.mul(1.0 + upper_interval_return),
            'lower_var_return': lower_var_return_with_current,
            'upper_var_return': upper_var_return_with_current,
            'lower_var_price': open_for_var.mul(1.0 + lower_var_return),
            'upper_var_price': open_for_var.mul(1.0 + upper_var_return),
            'lower_expected_shortfall_return': lower_expected_shortfall_return_with_current,
            'upper_expected_shortfall_return': upper_expected_shortfall_return_with_current,
            'lower_expected_shortfall_price': open_for_es.mul(1.0 + lower_expected_shortfall_return),
            'upper_expected_shortfall_price': open_for_es.mul(1.0 + upper_expected_shortfall_return),
            'lower_breaches': lower_breaches_with_current,
            'upper_breaches': upper_breaches_with_current,
            'either_side_breaches': either_side_breaches_with_current,
            'lower_rolling_breach_rate': lower_rolling_breach_rate,
            'upper_rolling_breach_rate': upper_rolling_breach_rate,
            'either_side_rolling_breach_rate': either_side_rolling_breach_rate,
            'lower_expected_breach_rate': lower_expected_breach_rate,
            'upper_expected_breach_rate': upper_expected_breach_rate,
            'either_side_expected_breach_rate': either_side_expected_breach_rate,
        }

    garch_trade_history_context = {
        'window': trade_range_garch_window,
        'horizon_sessions': trade_range_garch_default_horizon,
        'interval_confidence_levels': trade_range_garch_interval_levels,
        'tail_confidence_levels': trade_range_garch_tail_levels,
        'metrics_by_confidence': garch_history_metrics_by_confidence,
        'session_returns': garch_session_returns,
        'session_open': garch_trade_frame['Close'].shift(1).astype(float),
        'session_close': garch_trade_frame['Close'].astype(float),
    }
    garch_trade_history_fig = plot_trade_range_history_profile(
        history_context=garch_trade_history_context,
        ticker_label=f'{ticker_str} GARCH(1,1) Normal',
    )
    garch_trade_history_contexts_by_key = {
        (int(trade_range_garch_window), int(trade_range_garch_default_horizon)): garch_trade_history_context,
    }

    def build_garch_trade_range_history_context(window, horizon_sessions=1):
        window = int(window)
        horizon_sessions = int(horizon_sessions)
        history_cache_key = (window, horizon_sessions)
        cached_history_context = garch_trade_history_contexts_by_key.get(history_cache_key)
        if cached_history_context is not None:
            return cached_history_context

        if horizon_sessions == int(trade_range_garch_default_horizon):
            history_session_returns = garch_session_returns.copy()
            history_session_open = garch_trade_frame['Close'].shift(1).astype(float).reindex(history_session_returns.index)
            history_session_close = garch_trade_frame['Close'].astype(float).reindex(history_session_returns.index)
            history_fit_returns = garch_completed_returns.copy()
            include_current_session = True
        else:
            history_holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
                garch_trade_frame,
                horizon_sessions,
            )
            history_session_returns = history_holding_frame['session_return'].dropna()
            history_session_open = history_holding_frame['session_open'].astype(float).reindex(history_session_returns.index)
            history_session_close = history_holding_frame['session_close'].astype(float).reindex(history_session_returns.index)
            history_fit_returns = history_session_returns.copy()
            include_current_session = False

        if len(history_fit_returns) < 50:
            raise ValueError(
                f'The GARCH option needs at least 50 completed {horizon_sessions}-session returns to build the selected history view. Only {len(history_fit_returns)} are available.'
            )

        history_model = arch_model(
            history_fit_returns.mul(100.0),
            mean='Constant',
            vol='GARCH',
            p=1,
            q=1,
            dist='normal',
            rescale=False,
        )
        history_fit = history_model.fit(disp='off')
        history_mean_return = float(history_fit.params.get('mu', 0.0) / 100.0)
        history_sigma_series = pd.Series(
            history_fit.conditional_volatility / 100.0,
            index=history_fit_returns.index,
        )

        current_history_date = None
        current_history_return = None
        current_history_mean_return = None
        current_history_sigma_return = None
        if include_current_session:
            current_history_forecast = history_fit.forecast(horizon=1, reindex=False)
            current_history_mean_return = float(current_history_forecast.mean.iloc[-1, 0] / 100.0)
            current_history_sigma_return = float(np.sqrt(current_history_forecast.variance.iloc[-1, 0]) / 100.0)
            if not np.isfinite(current_history_sigma_return) or current_history_sigma_return <= 0:
                raise ValueError(
                    'The GARCH option could not produce a positive current-session GARCH volatility forecast for the selected history view.'
                )
            current_history_date = garch_trade_frame.index[-1]
            current_history_return = float(garch_session_returns.iloc[-1])

        history_metrics_by_confidence = {}
        for confidence in sorted(set(trade_range_garch_interval_levels).union(trade_range_garch_tail_levels)):
            alpha = 1.0 - confidence
            interval_alpha = (1.0 - confidence) / 2.0
            interval_z = float(norm.ppf(interval_alpha))
            z_alpha = float(norm.ppf(alpha))
            pdf_alpha = float(norm.pdf(z_alpha))

            lower_interval_return = history_mean_return + (history_sigma_series * interval_z)
            upper_interval_return = history_mean_return - (history_sigma_series * interval_z)
            lower_var_return = history_mean_return + (history_sigma_series * z_alpha)
            upper_var_return = history_mean_return - (history_sigma_series * z_alpha)
            lower_expected_shortfall_return = history_mean_return - (history_sigma_series * (pdf_alpha / alpha))
            upper_expected_shortfall_return = history_mean_return + (history_sigma_series * (pdf_alpha / alpha))

            if include_current_session:
                current_lower_var_return = current_history_mean_return + (current_history_sigma_return * z_alpha)
                current_upper_var_return = current_history_mean_return - (current_history_sigma_return * z_alpha)
                current_lower_expected_shortfall_return = (
                    current_history_mean_return - (current_history_sigma_return * (pdf_alpha / alpha))
                )
                current_upper_expected_shortfall_return = (
                    current_history_mean_return + (current_history_sigma_return * (pdf_alpha / alpha))
                )

                lower_var_return_for_plot = pd.concat([
                    lower_var_return,
                    pd.Series([current_lower_var_return], index=[current_history_date]),
                ]).sort_index()
                upper_var_return_for_plot = pd.concat([
                    upper_var_return,
                    pd.Series([current_upper_var_return], index=[current_history_date]),
                ]).sort_index()
                lower_expected_shortfall_return_for_plot = pd.concat([
                    lower_expected_shortfall_return,
                    pd.Series([current_lower_expected_shortfall_return], index=[current_history_date]),
                ]).sort_index()
                upper_expected_shortfall_return_for_plot = pd.concat([
                    upper_expected_shortfall_return,
                    pd.Series([current_upper_expected_shortfall_return], index=[current_history_date]),
                ]).sort_index()

                lower_breaches = history_fit_returns.lt(lower_var_return).astype(float)
                upper_breaches = history_fit_returns.gt(upper_var_return).astype(float)
                lower_breaches_for_plot = pd.concat([
                    lower_breaches,
                    pd.Series([float(current_history_return < current_lower_var_return)], index=[current_history_date]),
                ]).sort_index()
                upper_breaches_for_plot = pd.concat([
                    upper_breaches,
                    pd.Series([float(current_history_return > current_upper_var_return)], index=[current_history_date]),
                ]).sort_index()
            else:
                lower_var_return_for_plot = lower_var_return.copy()
                upper_var_return_for_plot = upper_var_return.copy()
                lower_expected_shortfall_return_for_plot = lower_expected_shortfall_return.copy()
                upper_expected_shortfall_return_for_plot = upper_expected_shortfall_return.copy()
                lower_breaches_for_plot = history_session_returns.lt(lower_var_return_for_plot).astype(float)
                upper_breaches_for_plot = history_session_returns.gt(upper_var_return_for_plot).astype(float)

            either_side_breaches_for_plot = lower_breaches_for_plot.add(upper_breaches_for_plot, fill_value=0.0).clip(upper=1.0)
            lower_rolling_breach_rate = lower_breaches_for_plot.rolling(window).mean().dropna()
            upper_rolling_breach_rate = upper_breaches_for_plot.rolling(window).mean().dropna()
            either_side_rolling_breach_rate = either_side_breaches_for_plot.rolling(window).mean().dropna()

            lower_expected_breach_rate = pd.Series(
                data=np.full(len(lower_rolling_breach_rate.index), alpha, dtype=float),
                index=lower_rolling_breach_rate.index,
            )
            upper_expected_breach_rate = pd.Series(
                data=np.full(len(upper_rolling_breach_rate.index), alpha, dtype=float),
                index=upper_rolling_breach_rate.index,
            )
            either_side_expected_breach_rate = pd.Series(
                data=np.full(len(either_side_rolling_breach_rate.index), min(1.0, 2.0 * alpha), dtype=float),
                index=either_side_rolling_breach_rate.index,
            )

            open_for_interval = history_session_open.reindex(lower_interval_return.index)
            open_for_var = history_session_open.reindex(lower_var_return_for_plot.index)
            open_for_es = history_session_open.reindex(lower_expected_shortfall_return_for_plot.index)

            history_metrics_by_confidence[confidence] = {
                'session_returns': history_session_returns,
                'session_open': history_session_open,
                'session_close': history_session_close,
                'lower_interval_return': lower_interval_return,
                'upper_interval_return': upper_interval_return,
                'lower_interval_price': open_for_interval.mul(1.0 + lower_interval_return),
                'upper_interval_price': open_for_interval.mul(1.0 + upper_interval_return),
                'lower_var_return': lower_var_return_for_plot,
                'upper_var_return': upper_var_return_for_plot,
                'lower_var_price': open_for_var.mul(1.0 + lower_var_return_for_plot),
                'upper_var_price': open_for_var.mul(1.0 + upper_var_return_for_plot),
                'lower_expected_shortfall_return': lower_expected_shortfall_return_for_plot,
                'upper_expected_shortfall_return': upper_expected_shortfall_return_for_plot,
                'lower_expected_shortfall_price': open_for_es.mul(1.0 + lower_expected_shortfall_return_for_plot),
                'upper_expected_shortfall_price': open_for_es.mul(1.0 + upper_expected_shortfall_return_for_plot),
                'lower_breaches': lower_breaches_for_plot,
                'upper_breaches': upper_breaches_for_plot,
                'either_side_breaches': either_side_breaches_for_plot,
                'lower_rolling_breach_rate': lower_rolling_breach_rate,
                'upper_rolling_breach_rate': upper_rolling_breach_rate,
                'either_side_rolling_breach_rate': either_side_rolling_breach_rate,
                'lower_expected_breach_rate': lower_expected_breach_rate,
                'upper_expected_breach_rate': upper_expected_breach_rate,
                'either_side_expected_breach_rate': either_side_expected_breach_rate,
            }

        history_context = {
            'window': window,
            'horizon_sessions': horizon_sessions,
            'interval_confidence_levels': trade_range_garch_interval_levels,
            'tail_confidence_levels': trade_range_garch_tail_levels,
            'metrics_by_confidence': history_metrics_by_confidence,
            'session_returns': history_session_returns,
            'session_open': history_session_open,
            'session_close': history_session_close,
        }
        garch_trade_history_contexts_by_key[history_cache_key] = history_context
        return history_context
    # Current-session forecast uses the selected trailing fit window so the cone can switch windows like Block 7.
    def build_garch_trade_range_cone_components(window, horizon_sessions=1):
        window = int(window)
        horizon_sessions = max(1, int(horizon_sessions))

        garch_fit_returns = garch_completed_returns.tail(window).dropna()
        garch_fit_window = len(garch_fit_returns)
        if garch_fit_window < 21:
            raise ValueError(
                f'The GARCH option needs at least 21 completed sessions inside the trailing fit window. Only {garch_fit_window} are available for the {window}-session view.'
            )

        garch_trade_model = arch_model(
            garch_fit_returns.mul(100.0),
            mean='Constant',
            vol='GARCH',
            p=1,
            q=1,
            dist='normal',
            rescale=False,
        )
        garch_trade_fit = garch_trade_model.fit(disp='off')
        garch_trade_forecast = garch_trade_fit.forecast(horizon=horizon_sessions, reindex=False)

        forecast_mean_values = np.asarray(garch_trade_forecast.mean.iloc[-1], dtype=float)
        forecast_variance_values = np.asarray(garch_trade_forecast.variance.iloc[-1], dtype=float)
        if len(forecast_mean_values) < horizon_sessions or len(forecast_variance_values) < horizon_sessions:
            raise ValueError(
                f'The GARCH option could not produce a complete {horizon_sessions}-session forecast for the {window}-session fit window.'
            )

        garch_mean_return = float(np.nansum(forecast_mean_values[:horizon_sessions]) / 100.0)
        garch_sigma_return = float(
            np.sqrt(np.clip(forecast_variance_values[:horizon_sessions], a_min=0.0, a_max=None).sum()) / 100.0
        )
        if not np.isfinite(garch_sigma_return) or garch_sigma_return <= 0:
            raise ValueError(
                f'The GARCH option could not produce a positive {horizon_sessions}-session volatility forecast for the {window}-session fit window.'
            )

        garch_holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
            garch_trade_frame,
            horizon_sessions,
        )
        garch_sample_returns = garch_holding_frame['session_return'].iloc[:-1].tail(window).dropna()
        garch_effective_window = len(garch_sample_returns)
        if garch_effective_window == 0:
            raise ValueError(
                f'The GARCH option could not build a completed {horizon_sessions}-session return sample for the {window}-session fit window.'
            )

        garch_anchor_price = float(risk_distribution_analytics._resolve_current_reference_price(garch_trade_frame))
        garch_latest_price = float(garch_trade_frame.attrs.get('current_session_latest_price', garch_trade_frame['Close'].iloc[-1]))
        garch_session_date = pd.Timestamp(garch_trade_frame.attrs.get('current_session_date', garch_trade_frame.index[-1]))
        garch_median_return = garch_mean_return
        garch_median_price = garch_anchor_price * (1.0 + garch_median_return)

        garch_model_summary = pd.DataFrame([
            {
                'Model': 'GARCH(1,1) Normal',
                'Fit Window': garch_fit_window,
                'Forecast Horizon': horizon_sessions,
                'Forecast Mean Return': garch_mean_return,
                'Forecast Sigma Return': garch_sigma_return,
                'Omega': float(garch_trade_fit.params.get('omega', np.nan)),
                'Alpha(1)': float(garch_trade_fit.params.get('alpha[1]', np.nan)),
                'Beta(1)': float(garch_trade_fit.params.get('beta[1]', np.nan)),
                'Alpha + Beta': float(garch_trade_fit.params.get('alpha[1]', np.nan) + garch_trade_fit.params.get('beta[1]', np.nan)),
            }
        ])

        garch_interval_map = {}
        garch_range_rows = []
        for confidence in trade_range_garch_interval_levels:
            tail_probability = (1.0 - confidence) / 2.0
            lower_return = float(norm.ppf(tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
            upper_return = float(norm.ppf(1.0 - tail_probability, loc=garch_mean_return, scale=garch_sigma_return))
            lower_price = garch_anchor_price * (1.0 + lower_return)
            upper_price = garch_anchor_price * (1.0 + upper_return)
            garch_interval_map[confidence] = {
                'lower_return': lower_return,
                'upper_return': upper_return,
                'lower_price': lower_price,
                'upper_price': upper_price,
            }
            garch_range_rows.append(
                {
                    'Confidence': f'{confidence:.0%}',
                    'Lower Return': lower_return,
                    'Upper Return': upper_return,
                    'Lower Exit Price': lower_price,
                    'Upper Exit Price': upper_price,
                }
            )

        garch_long_tail_map = {}
        garch_short_tail_map = {}
        garch_tail_rows = []
        for confidence in trade_range_garch_tail_levels:
            alpha = 1.0 - confidence
            z_alpha = float(norm.ppf(alpha))
            pdf_alpha = float(norm.pdf(z_alpha))

            long_var_return = garch_mean_return + (garch_sigma_return * z_alpha)
            long_cvar_return = garch_mean_return - (garch_sigma_return * (pdf_alpha / alpha))
            short_var_return = garch_mean_return - (garch_sigma_return * z_alpha)
            short_cvar_return = garch_mean_return + (garch_sigma_return * (pdf_alpha / alpha))

            garch_long_tail_map[confidence] = {
                'var_return': long_var_return,
                'var_price': garch_anchor_price * (1.0 + long_var_return),
                'expected_shortfall_return': long_cvar_return,
                'expected_shortfall_price': garch_anchor_price * (1.0 + long_cvar_return),
            }
            garch_short_tail_map[confidence] = {
                'var_return': short_var_return,
                'var_price': garch_anchor_price * (1.0 + short_var_return),
                'expected_shortfall_return': short_cvar_return,
                'expected_shortfall_price': garch_anchor_price * (1.0 + short_cvar_return),
            }
            garch_tail_rows.append(
                {
                    'Confidence': f'{confidence:.0%}',
                    'Long VaR Floor': garch_long_tail_map[confidence]['var_price'],
                    'Long CVaR Floor': garch_long_tail_map[confidence]['expected_shortfall_price'],
                    'Short VaR Ceiling': garch_short_tail_map[confidence]['var_price'],
                    'Short CVaR Ceiling': garch_short_tail_map[confidence]['expected_shortfall_price'],
                }
            )

        garch_range_summary = pd.DataFrame(garch_range_rows)
        garch_tail_summary = pd.DataFrame(garch_tail_rows).rename(columns={
            'Confidence': 'Tail Confidence',
        })

        garch_trade_range_context = {
            'session_date': garch_session_date,
            'window': window,
            'horizon_sessions': horizon_sessions,
            'effective_window': garch_effective_window,
            'anchor_price': garch_anchor_price,
            'latest_price': garch_latest_price,
            'sample_returns': garch_sample_returns,
            'interval_confidence_levels': trade_range_garch_interval_levels,
            'tail_confidence_levels': trade_range_garch_tail_levels,
            'intervals': garch_interval_map,
            'long_tail_levels': garch_long_tail_map,
            'short_tail_levels': garch_short_tail_map,
            'median_return': garch_median_return,
            'median_price': garch_median_price,
        }

        return {
            'context': garch_trade_range_context,
            'model_summary': garch_model_summary,
            'range_summary': garch_range_summary,
            'tail_summary': garch_tail_summary,
        }

    garch_cone_components_by_key = {
        (int(trade_range_garch_window), int(trade_range_garch_default_horizon)): build_garch_trade_range_cone_components(
            int(trade_range_garch_window),
            int(trade_range_garch_default_horizon),
        )
    }
    garch_default_components = garch_cone_components_by_key[
        (int(trade_range_garch_window), int(trade_range_garch_default_horizon))
    ]

    formatted_garch_model_summary = garch_default_components['model_summary'].copy()
    for column in ('Forecast Mean Return', 'Forecast Sigma Return'):
        if column in formatted_garch_model_summary.columns:
            formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Omega', 'Alpha(1)', 'Beta(1)', 'Alpha + Beta'):
        if column in formatted_garch_model_summary.columns:
            formatted_garch_model_summary[column] = formatted_garch_model_summary[column].map(
                lambda value: f'{value:.4f}' if pd.notna(value) else value
            )

    garch_range_summary = garch_default_components['range_summary'].copy()
    if not garch_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in garch_range_summary.columns:
                garch_range_summary[column] = garch_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in garch_range_summary.columns:
                garch_range_summary[column] = garch_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    garch_tail_summary = garch_default_components['tail_summary'].copy()
    if not garch_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in garch_tail_summary.columns:
                garch_tail_summary[column] = garch_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

except Exception as exc:
    garch_option_error = exc
    print(f'GARCH option unavailable: {exc}')



def _trade_range_horizon_label(horizon):
    horizon = int(horizon)
    if horizon == 0:
        return '0DTE same-session Open -> Close'
    return '1-session' if horizon == 1 else f'{horizon}-session'


def _trade_range_history_basis(horizon):
    horizon = int(horizon)
    if horizon == 0:
        return 'completed same-session open-to-close return history'
    if horizon == 1:
        return 'completed close-to-close return history'
    return f'completed {horizon}-session close-based holding-period returns'


def _trade_range_confidence_label(confidence):
    return f'{float(confidence):.0%}'


def _trade_range_confidence_key(levels, confidence):
    target = float(confidence)
    for level in levels:
        if np.isclose(float(level), target):
            return level
    raise KeyError(f'Trade-range confidence {target:.0%} is not available.')


def _coerce_trade_range_confidence(confidence):
    if confidence in (None, ''):
        return float(trade_range_default_confidence)
    return float(_trade_range_confidence_key(trade_range_confidence_options, confidence))


def _filter_trade_range_table_by_confidence(frame, confidence, column_name=None):
    if frame is None:
        return frame
    filtered_frame = frame.copy()
    if filtered_frame.empty:
        return filtered_frame
    selected_label = _trade_range_confidence_label(confidence)
    candidate_columns = [column_name] if column_name is not None else ['Confidence', 'Tail Confidence']
    for candidate_column in candidate_columns:
        if candidate_column in filtered_frame.columns:
            return filtered_frame.loc[
                filtered_frame[candidate_column].eq(selected_label)
            ].reset_index(drop=True)
    return filtered_frame


def _filter_trade_range_history_context(history_context, confidence):
    selected_confidence = _trade_range_confidence_key(
        history_context['metrics_by_confidence'].keys(),
        confidence,
    )
    filtered_history_context = dict(history_context)
    filtered_history_context['interval_confidence_levels'] = [float(selected_confidence)]
    filtered_history_context['tail_confidence_levels'] = [float(selected_confidence)]
    filtered_history_context['metrics_by_confidence'] = {
        selected_confidence: history_context['metrics_by_confidence'][selected_confidence]
    }
    return filtered_history_context


def _filter_trade_range_cone_context(cone_context, confidence):
    selected_confidence = _trade_range_confidence_key(cone_context['intervals'].keys(), confidence)
    filtered_cone_context = dict(cone_context)
    filtered_cone_context['interval_confidence_levels'] = [float(selected_confidence)]
    filtered_cone_context['tail_confidence_levels'] = [float(selected_confidence)]
    filtered_cone_context['intervals'] = {
        selected_confidence: cone_context['intervals'][selected_confidence]
    }
    filtered_cone_context['long_tail_levels'] = {
        selected_confidence: cone_context['long_tail_levels'][selected_confidence]
    }
    filtered_cone_context['short_tail_levels'] = {
        selected_confidence: cone_context['short_tail_levels'][selected_confidence]
    }
    if 'range_summary_table' in cone_context:
        filtered_cone_context['range_summary_table'] = _filter_trade_range_table_by_confidence(
            cone_context['range_summary_table'],
            selected_confidence,
            'Confidence',
        )
    if 'tail_summary_table' in cone_context:
        filtered_cone_context['tail_summary_table'] = _filter_trade_range_table_by_confidence(
            cone_context['tail_summary_table'],
            selected_confidence,
            'Confidence',
        )
    return filtered_cone_context


def _filter_garch_trade_range_components(components, confidence):
    filtered_context = _filter_trade_range_cone_context(components['context'], confidence)
    selected_confidence = filtered_context['interval_confidence_levels'][0]
    return {
        'context': filtered_context,
        'model_summary': components['model_summary'].copy(),
        'range_summary': _filter_trade_range_table_by_confidence(
            components['range_summary'],
            selected_confidence,
            'Confidence',
        ),
        'tail_summary': _filter_trade_range_table_by_confidence(
            components['tail_summary'],
            selected_confidence,
            'Tail Confidence',
        ),
    }


def _format_empirical_trade_range_tables(cone_context):
    formatted_range_summary = cone_context['range_summary_table'].copy().rename(columns={
        'Lower Close': 'Lower Exit Price',
        'Upper Close': 'Upper Exit Price',
    })
    formatted_tail_summary = cone_context['tail_summary_table'].copy().rename(columns={
        'Confidence': 'Tail Confidence',
    })

    if not formatted_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    if not formatted_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in formatted_tail_summary.columns:
                formatted_tail_summary[column] = formatted_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    return formatted_range_summary, formatted_tail_summary


def _format_garch_trade_range_tables(components):
    formatted_model_summary = components['model_summary'].copy()
    for column in ('Forecast Mean Return', 'Forecast Sigma Return'):
        if column in formatted_model_summary.columns:
            formatted_model_summary[column] = formatted_model_summary[column].map(
                lambda value: f'{value:.2%}' if pd.notna(value) else value
            )
    for column in ('Omega', 'Alpha(1)', 'Beta(1)', 'Alpha + Beta'):
        if column in formatted_model_summary.columns:
            formatted_model_summary[column] = formatted_model_summary[column].map(
                lambda value: f'{value:.4f}' if pd.notna(value) else value
            )

    formatted_range_summary = components['range_summary'].copy()
    if not formatted_range_summary.empty:
        for column in ('Lower Return', 'Upper Return'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'{value:.2%}' if pd.notna(value) else value
                )
        for column in ('Lower Exit Price', 'Upper Exit Price'):
            if column in formatted_range_summary.columns:
                formatted_range_summary[column] = formatted_range_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    formatted_tail_summary = components['tail_summary'].copy()
    if not formatted_tail_summary.empty:
        for column in (
            'Long VaR Floor',
            'Long CVaR Floor',
            'Short VaR Ceiling',
            'Short CVaR Ceiling',
        ):
            if column in formatted_tail_summary.columns:
                formatted_tail_summary[column] = formatted_tail_summary[column].map(
                    lambda value: f'${value:,.2f}' if pd.notna(value) else value
                )

    return formatted_model_summary, formatted_range_summary, formatted_tail_summary


def _strip_trade_range_dropdowns(fig):
    stripped_fig = _copy.deepcopy(fig)
    stripped_fig.update_layout(updatemenus=[])
    return stripped_fig


def _build_empirical_trade_range_view(window, horizon, confidence=None):
    window = int(window)
    horizon = int(horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    cached_history_context = trade_range_history_contexts_by_horizon.get(horizon)
    if cached_history_context is not None and window in cached_history_context['metrics_by_window']:
        history_context = {
            'window': window,
            'horizon_sessions': horizon,
            'interval_confidence_levels': cached_history_context['interval_confidence_levels'],
            'tail_confidence_levels': cached_history_context['tail_confidence_levels'],
            'metrics_by_confidence': cached_history_context['metrics_by_window'][window],
            'return_basis': cached_history_context.get('return_basis'),
            'session_returns': cached_history_context['session_returns'],
            'session_open': cached_history_context['session_open'],
            'session_close': cached_history_context['session_close'],
        }
    else:
        history_context = risk_distribution_analytics.build_trade_range_history_context(
            price_frame=trade_range_price_frame,
            windows=[window],
            horizon_sessions=horizon,
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
            default_window=window,
        )
        history_context = {
            'window': window,
            'horizon_sessions': horizon,
            'interval_confidence_levels': history_context['interval_confidence_levels'],
            'tail_confidence_levels': history_context['tail_confidence_levels'],
            'metrics_by_confidence': history_context['metrics_by_confidence'],
            'return_basis': history_context.get('return_basis'),
            'session_returns': history_context['session_returns'],
            'session_open': history_context['session_open'],
            'session_close': history_context['session_close'],
        }
    history_context = _filter_trade_range_history_context(history_context, confidence)
    history_fig = _strip_trade_range_dropdowns(
        plot_trade_range_history_profile(
            history_context=history_context,
            ticker_label=ticker_str,
        )
    )
    cone_cache_key = (window, horizon)
    cone_context = trade_range_cone_contexts_by_key.get(cone_cache_key)
    if cone_context is None:
        cone_context = risk_distribution_analytics.build_trade_range_probability_context(
            price_frame=trade_range_price_frame,
            window=window,
            horizon_sessions=horizon,
            interval_confidence_levels=trade_range_interval_levels,
            tail_confidence_levels=trade_range_tail_levels,
        )
        trade_range_cone_contexts_by_key[cone_cache_key] = cone_context
    filtered_cone_context = _filter_trade_range_cone_context(cone_context, confidence)
    range_summary_window, tail_summary_window = _format_empirical_trade_range_tables(filtered_cone_context)
    cone_fig = _strip_trade_range_dropdowns(
        plot_trade_range_probability_cone(
            cone_context=filtered_cone_context,
            ticker_label=ticker_str,
        )
    )
    combined_fig = plot_trade_range_stack_view(
        history_fig,
        cone_fig,
        title_text=(
            f'{ticker_str} Historical Two-Sided Trade Range Profile and Current Cone '
            f'({int(window)}-Session Lookback, {_trade_range_horizon_label(horizon)} Horizon)'
        ),
    )
    combined_fig.update_layout(updatemenus=[])
    return {
        'caption': (
            f'Empirical trade range using {_trade_range_history_basis(horizon)} '
            f'at {_trade_range_confidence_label(confidence)} confidence.'
        ),
        'model_summary': None,
        'range_summary': range_summary_window,
        'tail_summary': tail_summary_window,
        'cone_figure': cone_fig,
        'figure': combined_fig,
    }


def _build_garch_trade_range_view(window, horizon, confidence=None):
    window = int(window)
    horizon = int(horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    garch_cache_key = (window, horizon)
    if garch_cache_key not in garch_cone_components_by_key:
        garch_cone_components_by_key[garch_cache_key] = build_garch_trade_range_cone_components(window, horizon)
    components = _filter_garch_trade_range_components(
        garch_cone_components_by_key[garch_cache_key],
        confidence,
    )
    formatted_model_summary_window, range_summary_window, tail_summary_window = _format_garch_trade_range_tables(components)
    history_context = _filter_trade_range_history_context(
        build_garch_trade_range_history_context(window, horizon),
        confidence,
    )
    history_fig = _strip_trade_range_dropdowns(
        plot_trade_range_history_profile(
            history_context=history_context,
            ticker_label=f'{ticker_str} GARCH(1,1) Normal',
        )
    )
    cone_fig = _strip_trade_range_dropdowns(
        plot_trade_range_probability_cone(
            cone_context=components['context'],
            ticker_label=f'{ticker_str} GARCH(1,1)',
        )
    )
    combined_fig = plot_trade_range_stack_view(
        history_fig,
        cone_fig,
        title_text=(
            f'{ticker_str} GARCH(1,1) Historical Diagnostics and Current Cone '
            f'({int(window)}-Session Fit Window, {int(horizon)}-Session Horizon)'
        ),
    )
    combined_fig.update_layout(updatemenus=[])
    if horizon == 1:
        caption = (
            f'GARCH(1,1)-normal trade range fit on completed one-session returns at '
            f'{_trade_range_confidence_label(confidence)} confidence. '
            'Session lookback changes the current-session cone, and the historical diagnostics stay on the same one-session basis.'
        )
    else:
        caption = (
            f'GARCH(1,1)-normal view filtered to {_trade_range_confidence_label(confidence)} confidence. '
            f'GARCH(1,1)-normal current cones still aggregate one-session forecasts across the selected {horizon}-session horizon. '
            f'The historical diagnostics now plot completed {horizon}-session holding returns through a horizon-matched GARCH filter so the rolling return panel stays aligned with the selected horizon.'
        )
    return {
        'caption': caption,
        'model_summary': formatted_model_summary_window,
        'range_summary': range_summary_window,
        'tail_summary': tail_summary_window,
        'cone_figure': cone_fig,
        'figure': combined_fig,
    }


trade_range_view_builders = {
    'Empirical': _build_empirical_trade_range_view,
}
trade_range_window_options_by_method = {
    'Empirical': list(trade_range_window_options),
}

if all(name in globals() for name in ('garch_trade_history_fig', 'garch_cone_components_by_key')):
    trade_range_view_builders['GARCH(1,1) Normal'] = _build_garch_trade_range_view
    trade_range_window_options_by_method['GARCH(1,1) Normal'] = list(trade_range_garch_window_options)

trade_range_view_cache = {}


def get_trade_range_view(method, window, horizon, confidence=None):
    confidence = _coerce_trade_range_confidence(confidence)
    cache_key = (str(method), int(window), int(horizon), float(confidence))
    if cache_key not in trade_range_view_cache:
        trade_range_view_cache[cache_key] = trade_range_view_builders[str(method)](
            int(window),
            int(horizon),
            float(confidence),
        )
    return trade_range_view_cache[cache_key]


def _set_default_trade_range_outputs():
    default_empirical_view = get_trade_range_view(
        'Empirical',
        int(trade_range_default_window),
        int(trade_range_default_horizon),
        float(trade_range_default_confidence),
    )
    globals()['range_summary'] = default_empirical_view['range_summary'].copy()
    globals()['tail_summary'] = default_empirical_view['tail_summary'].copy()
    globals()['trade_range_fig'] = default_empirical_view['cone_figure']
    globals()['trade_range_combined_fig'] = default_empirical_view['figure']

    if 'GARCH(1,1) Normal' in trade_range_view_builders:
        default_garch_window = trade_range_garch_window if 'trade_range_garch_window' in globals() else trade_range_window_options_by_method['GARCH(1,1) Normal'][0]
        default_garch_view = get_trade_range_view(
            'GARCH(1,1) Normal',
            int(default_garch_window),
            int(trade_range_garch_default_horizon if 'trade_range_garch_default_horizon' in globals() else 1),
            float(trade_range_default_confidence),
        )
        globals()['formatted_garch_model_summary'] = default_garch_view['model_summary'].copy()
        globals()['garch_range_summary'] = default_garch_view['range_summary'].copy()
        globals()['garch_tail_summary'] = default_garch_view['tail_summary'].copy()
        globals()['garch_trade_range_fig'] = default_garch_view['cone_figure']
        globals()['garch_trade_combined_fig'] = default_garch_view['figure']


from dash import Dash, Input, Output, State, dash_table, dcc, html
import socket
import uuid

required_trade_range_globals = (
    'get_trade_range_view',
    'trade_range_window_options_by_method',
    'trade_range_default_window',
    'trade_range_default_horizon',
)
missing_trade_range_globals = [name for name in required_trade_range_globals if name not in globals()]
if missing_trade_range_globals:
    raise NameError(
        'Run Block 7 before Block 7B. Missing globals: ' + ', '.join(missing_trade_range_globals)
    )



_set_default_trade_range_outputs()

def _dash_trade_range_window_bounds(method):
    method = str(method)
    min_window = 21
    if method == 'GARCH(1,1) Normal' and 'garch_completed_returns' in globals():
        max_window = int(len(garch_completed_returns))
    else:
        max_window = int(len(trade_range_price_frame.dropna()) - 1)
    max_window = max(min_window, max_window)
    return min_window, max_window


def _dash_trade_range_preferred_window(method):
    method = str(method)
    preferred_window = int(trade_range_default_window)
    if method == 'GARCH(1,1) Normal' and 'trade_range_garch_window' in globals():
        preferred_window = int(trade_range_garch_window)
    min_window, max_window = _dash_trade_range_window_bounds(method)
    return max(min_window, min(max_window, preferred_window))


def _coerce_dash_trade_range_window(method, value):
    min_window, max_window = _dash_trade_range_window_bounds(method)
    preferred_window = _dash_trade_range_preferred_window(method)
    helper_text = f'Available range: {min_window} to {max_window} completed sessions.'

    if value in (None, ''):
        return preferred_window, min_window, max_window, helper_text

    try:
        normalized_value = int(value)
    except (TypeError, ValueError):
        return (
            preferred_window,
            min_window,
            max_window,
            f'{helper_text} Using {preferred_window}-session lookback because the input was not a whole number.',
        )

    if normalized_value < min_window:
        return (
            min_window,
            min_window,
            max_window,
            f'{helper_text} Using {min_window}-session lookback because the requested value was below the supported minimum.',
        )

    if normalized_value > max_window:
        return (
            max_window,
            min_window,
            max_window,
            f'{helper_text} Using {max_window}-session lookback because the request exceeded the available completed-session history.',
        )

    return normalized_value, min_window, max_window, helper_text


def _dash_trade_range_default_window(method, current_window=None):
    if current_window is None:
        return _dash_trade_range_preferred_window(method)
    return _coerce_dash_trade_range_window(method, current_window)[0]


def _dash_trade_range_horizon_bounds(method, window):
    method = str(method)
    window, _, _, _ = _coerce_dash_trade_range_window(method, window)
    available_sessions = int(len(trade_range_price_frame.dropna()))
    max_horizon = max(1, (available_sessions - int(window) + 1) // 2)
    min_horizon = 1 if method == 'GARCH(1,1) Normal' else 0
    return min_horizon, max_horizon


def _dash_trade_range_preferred_horizon(method, window):
    min_horizon, max_horizon = _dash_trade_range_horizon_bounds(method, window)
    preferred_horizon = int(trade_range_default_horizon)
    if method == 'GARCH(1,1) Normal':
        preferred_horizon = int(globals().get('trade_range_garch_default_horizon', 1))
    return max(min_horizon, min(max_horizon, preferred_horizon))


def _coerce_dash_trade_range_horizon(method, window, value):
    min_horizon, max_horizon = _dash_trade_range_horizon_bounds(method, window)
    preferred_horizon = _dash_trade_range_preferred_horizon(method, window)
    if min_horizon == 0:
        helper_text = (
            f'Available range: 0 to {max_horizon} sessions. '
            '0 means 0DTE same-session Open -> Close; 1+ means close-to-close holding periods.'
        )
    else:
        helper_text = (
            f'Available range: {min_horizon} to {max_horizon} sessions. '
            '1 session means close-to-close.'
        )

    if value in (None, ''):
        return preferred_horizon, min_horizon, max_horizon, helper_text

    try:
        normalized_value = int(value)
    except (TypeError, ValueError):
        return (
            preferred_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {preferred_horizon}-session horizon because the input was not a whole number.',
        )

    if normalized_value < min_horizon:
        return (
            min_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {min_horizon}-session horizon because the requested value was below the supported minimum.',
        )

    if normalized_value > max_horizon:
        return (
            max_horizon,
            min_horizon,
            max_horizon,
            f'{helper_text} Using {max_horizon}-session horizon because the request exceeded the available completed-session history.',
        )

    return normalized_value, min_horizon, max_horizon, helper_text


def _dash_trade_range_default_horizon(method, window, current_horizon=None):
    if current_horizon is None:
        return _dash_trade_range_preferred_horizon(method, window)
    return _coerce_dash_trade_range_horizon(method, window, current_horizon)[0]


def _dash_trade_range_snapshot_table(method, window, horizon, confidence, view):
    rows = [
        {'Metric': 'Method', 'Value': method},
        {'Metric': 'Session Lookback Window', 'Value': f'{int(window)}-session'},
        {'Metric': 'Holding Horizon', 'Value': _trade_range_horizon_label(horizon)},
        {'Metric': 'Confidence Level', 'Value': _trade_range_confidence_label(confidence)},
        {'Metric': 'Description', 'Value': view['caption']},
    ]

    if method == 'GARCH(1,1) Normal' and view['model_summary'] is not None and not view['model_summary'].empty:
        summary_row = view['model_summary'].iloc[0]
        preferred_fields = [
            ('Forecast Mean', 'Forecast Mean Return'),
            ('Forecast Sigma', 'Forecast Sigma Return'),
            ('Omega', 'Omega'),
            ('Alpha(1)', 'Alpha(1)'),
            ('Beta(1)', 'Beta(1)'),
            ('Alpha + Beta', 'Alpha + Beta'),
        ]
        for label, column in preferred_fields:
            if column in summary_row.index:
                rows.append({'Metric': label, 'Value': str(summary_row[column])})
    else:
        rows.append({'Metric': 'Data Basis', 'Value': _trade_range_history_basis(horizon)})

    return pd.DataFrame(rows)


def _dash_table_payload(frame):
    if frame is None or frame.empty:
        frame = pd.DataFrame({'Info': ['No data available']})
    frame = frame.fillna('')
    columns = [{'name': column, 'id': column} for column in frame.columns]
    data = frame.astype(str).to_dict('records')
    return columns, data


def _find_open_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(('127.0.0.1', 0))
        sock.listen(1)
        return sock.getsockname()[1]


dash_trade_range_id = f"trade-range-dash-{uuid.uuid4().hex[:8]}"
dash_trade_range_methods = [str(method) for method in trade_range_window_options_by_method.keys()]
dash_trade_range_default_method = 'Empirical' if 'Empirical' in dash_trade_range_methods else dash_trade_range_methods[0]
dash_trade_range_default_window = _dash_trade_range_default_window(dash_trade_range_default_method)
dash_trade_range_default_horizon = _dash_trade_range_default_horizon(
    dash_trade_range_default_method,
    dash_trade_range_default_window,
)
dash_trade_range_default_confidence = float(trade_range_default_confidence)

dash_trade_range_app = Dash(dash_trade_range_id)
dash_trade_range_app.layout = html.Div(
    [
        html.Div(
            [
                html.Div(
                    [
                        html.Label('Method', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Dropdown(
                            id=f'{dash_trade_range_id}-method',
                            options=[{'label': method, 'value': method} for method in dash_trade_range_methods],
                            value=dash_trade_range_default_method,
                            clearable=False,
                        ),
                    ],
                    style={'flex': '1 1 320px', 'minWidth': '260px'},
                ),
                html.Div(
                    [
                        html.Label('Session Lookback Window', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Input(
                            id=f'{dash_trade_range_id}-window',
                            type='number',
                            value=dash_trade_range_default_window,
                            min=_dash_trade_range_window_bounds(dash_trade_range_default_method)[0],
                            max=_dash_trade_range_window_bounds(dash_trade_range_default_method)[1],
                            step=1,
                            debounce=True,
                            inputMode='numeric',
                            style={'width': '100%'},
                        ),
                        html.Div(
                            id=f'{dash_trade_range_id}-window-status',
                            style={'marginTop': '6px', 'fontSize': '12px', 'color': '#94a3b8'},
                        ),
                    ],
                    style={'flex': '0 0 220px', 'minWidth': '180px'},
                ),
                html.Div(
                    [
                        html.Label('Holding Horizon (0 = 0DTE Open -> Close)', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Input(
                            id=f'{dash_trade_range_id}-horizon',
                            type='number',
                            value=dash_trade_range_default_horizon,
                            min=_dash_trade_range_horizon_bounds(
                                dash_trade_range_default_method,
                                dash_trade_range_default_window,
                            )[0],
                            max=_dash_trade_range_horizon_bounds(
                                dash_trade_range_default_method,
                                dash_trade_range_default_window,
                            )[1],
                            step=1,
                            debounce=True,
                            inputMode='numeric',
                            style={'width': '100%'},
                        ),
                        html.Div(
                            id=f'{dash_trade_range_id}-horizon-status',
                            style={'marginTop': '6px', 'fontSize': '12px', 'color': '#94a3b8'},
                        ),
                    ],
                    style={'flex': '0 0 220px', 'minWidth': '180px'},
                ),
                html.Div(
                    [
                        html.Label('Confidence', style={'display': 'block', 'marginBottom': '6px', 'fontWeight': '600'}),
                        dcc.Dropdown(
                            id=f'{dash_trade_range_id}-confidence',
                            options=[
                                {
                                    'label': _trade_range_confidence_label(confidence),
                                    'value': float(confidence),
                                }
                                for confidence in trade_range_confidence_options
                            ],
                            value=dash_trade_range_default_confidence,
                            clearable=False,
                        ),
                    ],
                    style={'flex': '0 0 200px', 'minWidth': '180px'},
                ),
            ],
            style={'display': 'flex', 'gap': '12px', 'flexWrap': 'wrap', 'marginBottom': '14px'},
        ),
        html.Div(id=f'{dash_trade_range_id}-caption', style={'marginBottom': '14px', 'fontWeight': '600'}),
        html.Div(
            [
                html.Div('Model Snapshot', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-snapshot-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        html.Div(
            [
                html.Div('Projected Return / Price Range Summary', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-range-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        html.Div(
            [
                html.Div('Projected Tail Summary', style={'marginBottom': '8px', 'fontWeight': '600'}),
                dash_table.DataTable(
                    id=f'{dash_trade_range_id}-tail-table',
                    style_table={'overflowX': 'auto', 'width': '100%'},
                    style_cell={
                        'textAlign': 'left',
                        'padding': '6px 10px',
                        'backgroundColor': '#0f172a',
                        'color': '#e2e8f0',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                        'whiteSpace': 'normal',
                        'height': 'auto',
                    },
                    style_header={
                        'backgroundColor': '#1e293b',
                        'color': '#f8fafc',
                        'fontWeight': '700',
                        'border': '1px solid rgba(148, 163, 184, 0.35)',
                    },
                ),
            ],
            style={'marginBottom': '18px'},
        ),
        dcc.Graph(
            id=f'{dash_trade_range_id}-figure',
            style={'width': '100%', 'height': '3000px'},
            config={'responsive': True, 'displaylogo': False},
        ),
    ],
    style={
        'width': '100%',
        'maxWidth': '100%',
        'padding': '4px 0 16px 0',
        'fontFamily': 'Inter, Segoe UI, sans-serif',
    },
)


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-window', 'value'),
    Output(f'{dash_trade_range_id}-window', 'min'),
    Output(f'{dash_trade_range_id}-window', 'max'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    State(f'{dash_trade_range_id}-window', 'value'),
)
def _sync_dash_trade_range_window(method, current_window):
    selected_window, min_window, max_window, _ = _coerce_dash_trade_range_window(method, current_window)
    return selected_window, min_window, max_window


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-horizon', 'value'),
    Output(f'{dash_trade_range_id}-horizon', 'min'),
    Output(f'{dash_trade_range_id}-horizon', 'max'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    Input(f'{dash_trade_range_id}-window', 'value'),
    State(f'{dash_trade_range_id}-horizon', 'value'),
)
def _sync_dash_trade_range_horizon(method, window, current_horizon):
    selected_window, _, _, _ = _coerce_dash_trade_range_window(method, window)
    selected_horizon, min_horizon, max_horizon, _ = _coerce_dash_trade_range_horizon(
        method,
        selected_window,
        current_horizon,
    )
    return selected_horizon, min_horizon, max_horizon


@dash_trade_range_app.callback(
    Output(f'{dash_trade_range_id}-caption', 'children'),
    Output(f'{dash_trade_range_id}-window-status', 'children'),
    Output(f'{dash_trade_range_id}-horizon-status', 'children'),
    Output(f'{dash_trade_range_id}-snapshot-table', 'columns'),
    Output(f'{dash_trade_range_id}-snapshot-table', 'data'),
    Output(f'{dash_trade_range_id}-range-table', 'columns'),
    Output(f'{dash_trade_range_id}-range-table', 'data'),
    Output(f'{dash_trade_range_id}-tail-table', 'columns'),
    Output(f'{dash_trade_range_id}-tail-table', 'data'),
    Output(f'{dash_trade_range_id}-figure', 'figure'),
    Input(f'{dash_trade_range_id}-method', 'value'),
    Input(f'{dash_trade_range_id}-window', 'value'),
    Input(f'{dash_trade_range_id}-horizon', 'value'),
    Input(f'{dash_trade_range_id}-confidence', 'value'),
)
def _render_dash_trade_range_view(method, window, horizon, confidence):
    method = str(method)
    window, _, _, window_status = _coerce_dash_trade_range_window(method, window)
    horizon, _, _, horizon_status = _coerce_dash_trade_range_horizon(method, window, horizon)
    confidence = _coerce_trade_range_confidence(confidence)
    view = get_trade_range_view(method, window, horizon, confidence)

    snapshot_columns, snapshot_data = _dash_table_payload(
        _dash_trade_range_snapshot_table(method, window, horizon, confidence, view)
    )
    range_columns, range_data = _dash_table_payload(view['range_summary'])
    tail_columns, tail_data = _dash_table_payload(view['tail_summary'])

    figure = _copy.deepcopy(view['figure'])
    figure.update_layout(autosize=True)

    caption = (
        f'{method} | {window}-session lookback | {_trade_range_horizon_label(horizon)} horizon | '
        f'{_trade_range_confidence_label(confidence)} confidence | {view["caption"]}'
    )
    return (
        caption,
        window_status,
        horizon_status,
        snapshot_columns,
        snapshot_data,
        range_columns,
        range_data,
        tail_columns,
        tail_data,
        figure,
    )


dash_trade_range_port = _find_open_port()
print(f'Dash Block 7B preview starting on port {dash_trade_range_port}.')
dash_trade_range_app.run(
    port=dash_trade_range_port,
    debug=False,
    jupyter_mode='inline',
    jupyter_width='100%',
    jupyter_height=3900,
    dev_tools_ui=False,
    dev_tools_props_check=False,
)

In [ ]:
# Block 7C: calibrate empirical breach rates across holding horizons and lookback windows

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


trade_range_breach_windows = [21, 50, 200]
trade_range_breach_confidence_levels = list(globals().get('trade_range_tail_levels', [0.95, 0.99]))
trade_range_breach_price_frame = globals().get(
    'trade_range_price_frame',
    globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].copy(),
)
trade_range_breach_complete_frame = (
    trade_range_breach_price_frame[['Open', 'Close']].dropna().sort_index().copy()
)
trade_range_breach_complete_rows = int(len(trade_range_breach_complete_frame))
if trade_range_breach_complete_rows < 2:
    raise ValueError('Block 7C requires at least two completed sessions with Open and Close prices.')

trade_range_breach_target_end = min(21, trade_range_breach_complete_rows)
trade_range_breach_horizon_step = 1
trade_range_breach_confidence_labels = {
    float(confidence): f'{confidence:.0%}' for confidence in trade_range_breach_confidence_levels
}
trade_range_breach_window_labels = [f'{int(window)}-session' for window in trade_range_breach_windows]
trade_range_breach_horizon_grid = sorted(
    {
        1,
        *range(
            min(trade_range_breach_horizon_step, trade_range_breach_target_end),
            trade_range_breach_target_end + 1,
            trade_range_breach_horizon_step,
        ),
        trade_range_breach_target_end,
    }
)
trade_range_breach_tick_step = 20 if trade_range_breach_target_end > 120 else 10
trade_range_breach_tick_values = [
    horizon
    for horizon in trade_range_breach_horizon_grid
    if horizon in (1, trade_range_breach_target_end) or horizon % trade_range_breach_tick_step == 0
]
trade_range_breach_cache_signature = (
    tuple(int(window) for window in trade_range_breach_windows),
    tuple(float(confidence) for confidence in trade_range_breach_confidence_levels),
    int(trade_range_breach_complete_rows),
    str(trade_range_breach_complete_frame.index[0]),
    str(trade_range_breach_complete_frame.index[-1]),
    float(trade_range_breach_complete_frame['Open'].iloc[0]),
    float(trade_range_breach_complete_frame['Open'].iloc[-1]),
    float(trade_range_breach_complete_frame['Close'].iloc[0]),
    float(trade_range_breach_complete_frame['Close'].iloc[-1]),
)
if globals().get('trade_range_breach_cache_signature') != trade_range_breach_cache_signature:
    trade_range_breach_snapshot_cache = {}
else:
    trade_range_breach_snapshot_cache = globals().get('trade_range_breach_snapshot_cache', {})
globals()['trade_range_breach_cache_signature'] = trade_range_breach_cache_signature


def _build_trade_range_breach_horizon_snapshot(horizon):
    horizon = int(horizon)
    if horizon <= 0:
        raise ValueError('horizon must be a positive integer.')
    if horizon > trade_range_breach_complete_rows:
        raise ValueError('horizon exceeds the available completed session count.')

    cached_snapshot = trade_range_breach_snapshot_cache.get(horizon)
    if isinstance(cached_snapshot, pd.DataFrame) and not cached_snapshot.empty:
        return cached_snapshot.copy()

    holding_frame = risk_distribution_analytics._build_session_holding_period_frame(
        trade_range_breach_complete_frame,
        horizon,
    )
    session_returns = holding_frame['session_return'].to_numpy(dtype=float)
    latest_calibration_date = pd.Timestamp(holding_frame.index[-1])

    records = []
    for window in trade_range_breach_windows:
        rolling_window = int(window)
        if len(session_returns) < rolling_window:
            rolling_samples = None
            valid_count = 0
            realized_returns = np.array([], dtype=float)
        else:
            rolling_samples = sliding_window_view(session_returns, rolling_window)
            valid_count = len(session_returns) - horizon - rolling_window + 1
            realized_returns = session_returns[horizon + rolling_window - 1:]

        for confidence in trade_range_breach_confidence_levels:
            alpha = 1.0 - float(confidence)
            actual_breach_rate = np.nan
            expected_breach_rate = np.nan
            calibration_date = pd.NaT

            if rolling_samples is not None and valid_count > 0:
                lower_var = np.quantile(rolling_samples, alpha, axis=1)[:valid_count]
                upper_var = np.quantile(rolling_samples, 1.0 - alpha, axis=1)[:valid_count]
                breaches = np.logical_or(
                    realized_returns < lower_var,
                    realized_returns > upper_var,
                ).astype(float)
                if len(breaches) >= rolling_window:
                    actual_breach_rate = float(breaches[-rolling_window:].mean())
                    expected_breach_rate = float(min(1.0, 2.0 * alpha))
                    calibration_date = latest_calibration_date

            records.append(
                {
                    'horizon': horizon,
                    'window': rolling_window,
                    'confidence': float(confidence),
                    'latest_calibration_date': calibration_date,
                    'actual_breach_rate': actual_breach_rate,
                    'expected_breach_rate': expected_breach_rate,
                    'excess_breach_rate': (
                        actual_breach_rate - expected_breach_rate
                        if pd.notna(actual_breach_rate) and pd.notna(expected_breach_rate)
                        else np.nan
                    ),
                }
            )

    snapshot_frame = pd.DataFrame(records)
    trade_range_breach_snapshot_cache[horizon] = snapshot_frame
    globals()['trade_range_breach_snapshot_cache'] = trade_range_breach_snapshot_cache
    return snapshot_frame.copy()


def _build_trade_range_breach_summary():
    summary_frames = [
        _build_trade_range_breach_horizon_snapshot(horizon)
        for horizon in trade_range_breach_horizon_grid
    ]
    summary = pd.concat(summary_frames, ignore_index=True).sort_values(
        ['confidence', 'window', 'horizon']
    ).reset_index(drop=True)

    support_text = (
        f'Displaying holding horizons 1 through {trade_range_breach_target_end} '
        f'in single-session steps across lookbacks {", ".join(str(window) for window in trade_range_breach_windows)}. '
        'Blank cells mean there was not enough completed history to measure the latest rolling breach rate '
        'for that horizon and lookback combination.'
    )
    return summary, list(trade_range_breach_horizon_grid), support_text


def _build_trade_range_breach_excess_figure(summary_frame, supported_horizons):
    return plot_trade_range_breach_excess_view(
        summary_frame,
        supported_horizons=supported_horizons,
        windows=trade_range_breach_windows,
        confidence_levels=trade_range_breach_confidence_levels,
        confidence_labels=trade_range_breach_confidence_labels,
        ticker_label=ticker_str,
        target_end=trade_range_breach_target_end,
    )


def _format_trade_range_breach_summary(summary_frame):
    formatted_summary = summary_frame.copy()
    formatted_summary['Holding Horizon'] = formatted_summary['horizon'].map(
        lambda value: f'{int(value)}-session'
    )
    formatted_summary['Session Lookback Window'] = formatted_summary['window'].map(
        lambda value: f'{int(value)}-session'
    )
    formatted_summary['Confidence'] = formatted_summary['confidence'].map(
        lambda value: f'{value:.0%}'
    )
    formatted_summary['Latest Calibration Date'] = formatted_summary['latest_calibration_date'].map(
        lambda value: pd.Timestamp(value).strftime('%Y-%m-%d') if pd.notna(value) else ''
    )
    for column in ('actual_breach_rate', 'expected_breach_rate', 'excess_breach_rate'):
        formatted_summary[column] = formatted_summary[column].map(
            lambda value: f'{value:.2%}' if pd.notna(value) else ''
        )
    return formatted_summary.rename(
        columns={
            'actual_breach_rate': 'Actual Either-Side Breach Rate',
            'expected_breach_rate': 'Expected Either-Side Breach Rate',
            'excess_breach_rate': 'Excess Either-Side Breach Rate',
        }
    )[
        [
            'Holding Horizon',
            'Session Lookback Window',
            'Confidence',
            'Latest Calibration Date',
            'Actual Either-Side Breach Rate',
            'Expected Either-Side Breach Rate',
            'Excess Either-Side Breach Rate',
        ]
    ]


trade_range_breach_calibration_summary, trade_range_breach_supported_horizons, trade_range_breach_support_text = (
    _build_trade_range_breach_summary()
)
trade_range_breach_calibration_summary_display = _format_trade_range_breach_summary(
    trade_range_breach_calibration_summary
)
trade_range_breach_excess_fig = _build_trade_range_breach_excess_figure(
    trade_range_breach_calibration_summary,
    trade_range_breach_supported_horizons,
)

globals()['trade_range_breach_supported_horizons'] = list(trade_range_breach_supported_horizons)
globals()['trade_range_breach_calibration_summary'] = trade_range_breach_calibration_summary.copy()
globals()['trade_range_breach_calibration_summary_display'] = (
    trade_range_breach_calibration_summary_display.copy()
)
globals()['trade_range_breach_excess_fig'] = trade_range_breach_excess_fig

print(trade_range_breach_support_text)
display(trade_range_breach_excess_fig)
display(trade_range_breach_calibration_summary_display)

In [ ]:
# Block 7D: plot average empirical breach rates across all holding horizons over time

import numpy as np
import pandas as pd

if 'trade_range_breach_calibration_summary' not in globals():
    raise ValueError('Run Block 7C before Block 7D to build the breach-rate calibration summary.')

trade_range_breach_average_price_frame = globals().get(
    'trade_range_breach_price_frame',
    globals().get('trade_range_price_frame', globals().get('ticker_trade_range_source', ticker)[['Open', 'Close']].copy()),
)
trade_range_breach_average_price_frame = (
    trade_range_breach_average_price_frame[['Open', 'Close']].dropna().sort_index().copy()
)
trade_range_breach_average_horizons = [
    int(horizon)
    for horizon in globals().get(
        'trade_range_breach_supported_horizons',
        sorted(int(value) for value in globals()['trade_range_breach_calibration_summary']['horizon'].dropna().unique()),
    )
]
trade_range_breach_average_windows = [
    int(window)
    for window in globals().get(
        'trade_range_breach_windows',
        sorted(int(value) for value in globals()['trade_range_breach_calibration_summary']['window'].dropna().unique()),
    )
]
trade_range_breach_average_confidences = [
    float(confidence)
    for confidence in globals().get(
        'trade_range_breach_confidence_levels',
        sorted(float(value) for value in globals()['trade_range_breach_calibration_summary']['confidence'].dropna().unique()),
    )
]
trade_range_breach_average_signature = (
    tuple(trade_range_breach_average_horizons),
    tuple(trade_range_breach_average_windows),
    tuple(trade_range_breach_average_confidences),
    int(len(trade_range_breach_average_price_frame)),
    str(trade_range_breach_average_price_frame.index[0]),
    str(trade_range_breach_average_price_frame.index[-1]),
    float(trade_range_breach_average_price_frame['Open'].iloc[0]),
    float(trade_range_breach_average_price_frame['Open'].iloc[-1]),
    float(trade_range_breach_average_price_frame['Close'].iloc[0]),
    float(trade_range_breach_average_price_frame['Close'].iloc[-1]),
)
if globals().get('trade_range_breach_average_signature') != trade_range_breach_average_signature:
    trade_range_breach_average_contexts_by_horizon = {}
else:
    trade_range_breach_average_contexts_by_horizon = globals().get(
        'trade_range_breach_average_contexts_by_horizon',
        {},
    )
globals()['trade_range_breach_average_signature'] = trade_range_breach_average_signature

trade_range_breach_average_palette = ['#38bdf8', '#22c55e', '#f59e0b', '#f97316', '#a855f7']
trade_range_breach_average_colors = {
    int(window): trade_range_breach_average_palette[index % len(trade_range_breach_average_palette)]
    for index, window in enumerate(trade_range_breach_average_windows)
}


def _build_trade_range_breach_average_context(horizon):
    horizon = int(horizon)
    cached_context = trade_range_breach_average_contexts_by_horizon.get(horizon)
    if isinstance(cached_context, dict) and cached_context:
        return cached_context

    history_context = risk_distribution_analytics.build_trade_range_history_context(
        price_frame=trade_range_breach_average_price_frame,
        windows=trade_range_breach_average_windows,
        horizon_sessions=horizon,
        interval_confidence_levels=trade_range_breach_average_confidences,
        tail_confidence_levels=trade_range_breach_average_confidences,
        default_window=trade_range_breach_average_windows[0],
    )
    trade_range_breach_average_contexts_by_horizon[horizon] = history_context
    globals()['trade_range_breach_average_contexts_by_horizon'] = trade_range_breach_average_contexts_by_horizon
    return history_context


def _build_trade_range_breach_average_panel():
    averaged_series = {}
    horizon_count_series = {}
    support_lines = []

    for confidence in trade_range_breach_average_confidences:
        for window in trade_range_breach_average_windows:
            horizon_series = []
            for horizon in trade_range_breach_average_horizons:
                history_context = _build_trade_range_breach_average_context(horizon)
                metric_set = history_context['metrics_by_window'][int(window)][float(confidence)]
                breach_rate_series = metric_set.get('either_side_rolling_breach_rate', pd.Series(dtype=float)).dropna()
                if breach_rate_series.empty:
                    continue
                horizon_series.append(
                    breach_rate_series.rename(f'h{int(horizon)}')
                )

            if not horizon_series:
                continue

            aligned_frame = pd.concat(horizon_series, axis=1, join='inner').dropna(how='any')
            if aligned_frame.empty:
                continue

            averaged_series[(float(confidence), int(window))] = aligned_frame.mean(axis=1)
            horizon_count_series[(float(confidence), int(window))] = pd.Series(
                data=np.full(len(aligned_frame.index), aligned_frame.shape[1], dtype=int),
                index=aligned_frame.index,
            )
            support_lines.append(
                f'{float(confidence):.0%} / {int(window)}-session starts {aligned_frame.index.min():%Y-%m-%d} '
                f'with all {aligned_frame.shape[1]} horizons aligned.'
            )

    if not averaged_series:
        raise ValueError('Unable to build any averaged breach-rate series across the configured horizons.')

    average_panel = pd.concat(averaged_series, axis=1).sort_index(axis=1)
    count_panel = pd.concat(horizon_count_series, axis=1).sort_index(axis=1)
    support_text = 'Averages are taken across all configured holding horizons on dates where every horizon-specific rolling breach-rate series is available. '
    support_text += ' '.join(sorted(set(support_lines)))
    return average_panel, count_panel, support_text


def _build_trade_range_breach_average_figure(average_panel, count_panel):
    return plot_trade_range_breach_average_view(
        average_panel,
        count_panel,
        horizons=trade_range_breach_average_horizons,
        windows=trade_range_breach_average_windows,
        confidence_levels=trade_range_breach_average_confidences,
        window_colors=trade_range_breach_average_colors,
        ticker_label=ticker_str,
    )


trade_range_breach_average_panel, trade_range_breach_average_count_panel, trade_range_breach_average_support_text = (
    _build_trade_range_breach_average_panel()
)
trade_range_breach_average_fig = _build_trade_range_breach_average_figure(
    trade_range_breach_average_panel,
    trade_range_breach_average_count_panel,
)
globals()['trade_range_breach_average_panel'] = trade_range_breach_average_panel.copy()
globals()['trade_range_breach_average_count_panel'] = trade_range_breach_average_count_panel.copy()
globals()['trade_range_breach_average_support_text'] = trade_range_breach_average_support_text
globals()['trade_range_breach_average_fig'] = trade_range_breach_average_fig

print(trade_range_breach_average_support_text)
display(trade_range_breach_average_fig)

In [ ]:
# Block 8: backtest fixed payout structures against the 95% long confidence-interval floor

strategy_payout_options = [(float(risk_dollars), float(100 - risk_dollars)) for risk_dollars in range(10, 100, 10)]
strategy_default_payout = (70.0, 30.0)
strategy_confidence = 0.95
strategy_rolling_pnl_window = 21


def format_strategy_payout_label(risk_dollars, reward_dollars):
    return f'Risk ${risk_dollars:,.0f} / Reward ${reward_dollars:,.0f}'


def build_fixed_payout_trade_profile(strategy_label, metrics_by_confidence, *, confidence, risk_dollars, reward_dollars, window_label, rolling_window):
    metric_set = metrics_by_confidence.get(confidence)
    if metric_set is None:
        raise KeyError(f'{strategy_label} does not contain a {confidence:.0%} long interval-floor series.')

    long_interval_floor = pd.Series(metric_set['lower_interval_price']).dropna()
    session_open = pd.Series(metric_set['session_open']).reindex(long_interval_floor.index)
    session_close = pd.Series(metric_set['session_close']).reindex(long_interval_floor.index)
    session_returns = pd.Series(metric_set['session_returns']).reindex(long_interval_floor.index)

    trade_frame = pd.DataFrame({
        'Open': session_open,
        'Close': session_close,
        'Close-to-Close Return': session_returns,
        'Long 95% Interval Floor': long_interval_floor,
    }).dropna()
    if trade_frame.empty:
        raise ValueError(f'{strategy_label} does not contain any fully aligned historical trades to evaluate.')

    breakeven_win_rate = risk_dollars / (risk_dollars + reward_dollars)
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)

    trade_frame['Win'] = trade_frame['Close'].ge(trade_frame['Long 95% Interval Floor'])
    trade_frame['Outcome'] = np.where(trade_frame['Win'], 'Win', 'Loss')
    trade_frame['Trade PnL'] = np.where(trade_frame['Win'], reward_dollars, -risk_dollars)
    trade_frame['Rolling PnL'] = trade_frame['Trade PnL'].rolling(rolling_window, min_periods=1).sum()
    trade_frame['Rolling Win Rate'] = trade_frame['Win'].rolling(rolling_window, min_periods=1).mean()
    trade_frame['Cumulative PnL'] = trade_frame['Trade PnL'].cumsum()
    trade_frame['Running Peak PnL'] = trade_frame['Cumulative PnL'].cummax()
    trade_frame['Drawdown'] = trade_frame['Cumulative PnL'] - trade_frame['Running Peak PnL']
    trade_frame['Payout Structure'] = payout_label

    gross_profit = float(trade_frame.loc[trade_frame['Trade PnL'] > 0, 'Trade PnL'].sum())
    gross_loss = float(-trade_frame.loc[trade_frame['Trade PnL'] < 0, 'Trade PnL'].sum())
    profit_factor = np.nan if gross_loss == 0 else gross_profit / gross_loss

    summary = {
        'Payout Structure': payout_label,
        'Strategy': strategy_label,
        'Session Lookback Window': window_label,
        'Confidence': f'{confidence:.0%}',
        'Trades': int(len(trade_frame)),
        'Wins': int(trade_frame['Win'].sum()),
        'Losses': int((~trade_frame['Win']).sum()),
        'Win Rate': float(trade_frame['Win'].mean()),
        'Breakeven Win Rate': float(breakeven_win_rate),
        'Average Trade PnL': float(trade_frame['Trade PnL'].mean()),
        'Latest Rolling PnL': float(trade_frame['Rolling PnL'].iloc[-1]),
        'Total PnL': float(trade_frame['Trade PnL'].sum()),
        'Profit Factor': profit_factor,
        'Worst Drawdown': float(trade_frame['Drawdown'].min()),
    }

    return {
        'label': strategy_label,
        'summary': summary,
        'trade_frame': trade_frame,
    }


strategy_include_garch = 'garch_trade_history_context' in globals()
if not strategy_include_garch:
    print('Run Block 7 first if you want the GARCH strategy included in this fixed-payout evaluation.')


def build_strategy_profiles_for_payout(risk_dollars, reward_dollars):
    strategy_profiles = []
    for window in trade_range_history_context['windows']:
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'Current Method ({window}-Session)',
                metrics_by_confidence=trade_range_history_context['metrics_by_window'][window],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=window,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    if strategy_include_garch:
        garch_window_label = garch_trade_history_context.get('window', trade_range_default_window)
        strategy_profiles.append(
            build_fixed_payout_trade_profile(
                strategy_label=f'GARCH(1,1) Normal ({garch_window_label}-Session)',
                metrics_by_confidence=garch_trade_history_context['metrics_by_confidence'],
                confidence=strategy_confidence,
                risk_dollars=risk_dollars,
                reward_dollars=reward_dollars,
                window_label=garch_window_label,
                rolling_window=strategy_rolling_pnl_window,
            )
        )

    return strategy_profiles


strategy_default_payout_label = format_strategy_payout_label(*strategy_default_payout)
strategy_profiles_by_payout = {}
strategy_summary_by_payout = {}
strategy_backtest_trade_logs_by_payout = {}
strategy_backtest_equity_curve_by_payout = {}
strategy_backtest_rolling_pnl_by_payout = {}

for risk_dollars, reward_dollars in strategy_payout_options:
    payout_label = format_strategy_payout_label(risk_dollars, reward_dollars)
    strategy_profiles = build_strategy_profiles_for_payout(risk_dollars, reward_dollars)
    strategy_profiles_by_payout[payout_label] = strategy_profiles
    strategy_summary_by_payout[payout_label] = pd.DataFrame([profile['summary'] for profile in strategy_profiles])
    strategy_backtest_trade_logs_by_payout[payout_label] = {
        profile['label']: profile['trade_frame'].copy()
        for profile in strategy_profiles
    }
    strategy_backtest_equity_curve_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Cumulative PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()
    strategy_backtest_rolling_pnl_by_payout[payout_label] = pd.concat(
        {profile['label']: profile['trade_frame']['Rolling PnL'] for profile in strategy_profiles},
        axis=1,
    ).sort_index()

strategy_summary_lookup = pd.concat(strategy_summary_by_payout.values(), ignore_index=True)
formatted_strategy_summary_lookup = strategy_summary_lookup.copy()
for column in ('Win Rate', 'Breakeven Win Rate'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'{value:.2%}' if pd.notna(value) else value
        )
for column in ('Average Trade PnL', 'Latest Rolling PnL', 'Total PnL', 'Worst Drawdown'):
    if column in formatted_strategy_summary_lookup.columns:
        formatted_strategy_summary_lookup[column] = formatted_strategy_summary_lookup[column].map(
            lambda value: f'${value:,.2f}' if pd.notna(value) else value
        )
if 'Profit Factor' in formatted_strategy_summary_lookup.columns:
    formatted_strategy_summary_lookup['Profit Factor'] = formatted_strategy_summary_lookup['Profit Factor'].map(
        lambda value: 'N/A' if pd.isna(value) else f'{value:.2f}'
    )

display(formatted_strategy_summary_lookup)

strategy_backtest_trade_logs = strategy_backtest_trade_logs_by_payout[strategy_default_payout_label]
strategy_backtest_equity_curve = strategy_backtest_equity_curve_by_payout[strategy_default_payout_label]
strategy_backtest_rolling_pnl = strategy_backtest_rolling_pnl_by_payout[strategy_default_payout_label]
strategy_trade_log_preview = pd.concat(
    [
        profile['trade_frame'].assign(Strategy=profile['label'])
        for profile in strategy_profiles_by_payout[strategy_default_payout_label]
    ],
    axis=0,
).reset_index(names='Date')
strategy_trade_log_preview = strategy_trade_log_preview[['Date', 'Payout Structure', 'Strategy', 'Close', 'Long 95% Interval Floor', 'Outcome', 'Trade PnL', 'Rolling PnL', 'Rolling Win Rate', 'Cumulative PnL', 'Drawdown']]
display(strategy_trade_log_preview.tail(15))

strategy_backtest_fig = plot_fixed_payout_strategy_backtest_view(
    strategy_profiles_by_payout,
    payout_options=strategy_payout_options,
    default_payout=strategy_default_payout,
    rolling_pnl_window=strategy_rolling_pnl_window,
    ticker_label=ticker_str,
)
show_plotly_figure(strategy_backtest_fig)

In [ ]:
# Block 9: compare annualized GARCH-family volatility models to annualized rolling volatility estimators

import sys

def _purge_stale_modules(prefixes):
    for prefix in prefixes:
        matching_modules = [
            name
            for name in list(sys.modules)
            if name == prefix or name.startswith(f"{prefix}.")
        ]
        for module_name in matching_modules:
            sys.modules.pop(module_name, None)

try:
    from arch import arch_model
except ModuleNotFoundError as exc:
    if exc.name != "arch":
        raise
    raise ImportError(
        "Block 9 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
    ) from exc
except Exception:
    _purge_stale_modules(("arch", "matplotlib"))
    try:
        from arch import arch_model
    except ModuleNotFoundError as exc:
        if exc.name != "arch":
            raise
        raise ImportError(
            "Block 9 requires the 'arch' package. Install it in the notebook kernel environment with `pip install arch`."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            "Block 9 could not import `arch` because the notebook kernel is holding a stale matplotlib state. Restart the kernel and rerun Block 2 if this persists."
        ) from exc

close_series = ticker["Close"]
ohlc_frame = ticker[["Open", "High", "Low", "Close"]].copy()
returns = close_series.pct_change()
garch_input = returns.dropna() * 100

log_hl = np.log(ohlc_frame["High"] / ohlc_frame["Low"])
log_ho = np.log(ohlc_frame["High"] / ohlc_frame["Open"])
log_lo = np.log(ohlc_frame["Low"] / ohlc_frame["Open"])
log_co = np.log(ohlc_frame["Close"] / ohlc_frame["Open"])
log_oc = np.log(ohlc_frame["Open"] / ohlc_frame["Close"].shift(1))
log_hc = np.log(ohlc_frame["High"] / ohlc_frame["Close"])
log_lc = np.log(ohlc_frame["Low"] / ohlc_frame["Close"])

garman_klass_variance = 0.5 * (log_hl ** 2) - ((2 * np.log(2)) - 1) * (log_co ** 2)
parkinson_variance = (log_hl ** 2) / (4 * np.log(2))
rs_variance = (log_hc * log_ho) + (log_lc * log_lo)

volatility_model_specs = [
    ("GARCH(1,1)", dict(vol="GARCH", p=1, q=1, o=0), "#111111", "solid"),
    ("EGARCH(1,1)", dict(vol="EGARCH", p=1, o=1, q=1), "#d62728", "dash"),
    ("GJR-GARCH(1,1)", dict(vol="GARCH", p=1, o=1, q=1), "#2ca02c", "dot"),
]

rolling_realized_vol_specs = [
    ("Close-to-Close", "close-to-close", "#1f77b4", "solid"),
    ("Parkinson", "parkinson", "#9467bd", "dash"),
    ("Yang-Zhang", "yang-zhang", "#ff7f0e", "dot"),
    ("Garman-Klass", "garman-klass", "#8c564b", "dashdot"),
    ("Rogers-Satchell", "rogers-satchell", "#17becf", "longdash"),
]

ewma_realized_vol_specs = [
    ("EWMA Close-to-Close", "close-to-close", "#1f77b4", "longdashdot"),
    ("EWMA Parkinson", "parkinson", "#9467bd", "longdashdot"),
    ("EWMA Yang-Zhang", "yang-zhang", "#ff7f0e", "longdashdot"),
    ("EWMA Garman-Klass", "garman-klass", "#8c564b", "longdashdot"),
    ("EWMA Rogers-Satchell", "rogers-satchell", "#17becf", "longdashdot"),
]

annualized_model_vols = {}
for model_name, model_kwargs, _, _ in volatility_model_specs:
    model_fit = arch_model(
        garch_input,
        mean="Zero",
        dist="normal",
        rescale=False,
        **model_kwargs,
    ).fit(disp="off")
    annualized_model_vol = (model_fit.conditional_volatility / 100.0) * np.sqrt(252)
    annualized_model_vol.name = f"Annualized {model_name}"
    annualized_model_vols[model_name] = annualized_model_vol

def compute_rolling_realized_vol_series(window, method):
    if method == "close-to-close":
        return returns.rolling(window).std() * np.sqrt(252)
    if method == "parkinson":
        rolling_variance = parkinson_variance.rolling(window=window).mean().clip(lower=0)
    elif method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)
        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.rolling(window=window).var()
        open_to_close_variance = log_co.rolling(window=window).var()
        rs_component = rs_variance.rolling(window=window).mean()
        rolling_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
    elif method == "garman-klass":
        rolling_variance = garman_klass_variance.rolling(window=window).mean().clip(lower=0)
    elif method == "rogers-satchell":
        rolling_variance = rs_variance.rolling(window=window).mean().clip(lower=0)
    else:
        raise ValueError(f"Unsupported rolling volatility method: {method}")

    return np.sqrt(rolling_variance) * np.sqrt(252)

def compute_ewma_realized_vol_series(window, method):
    alpha = 2.0 / (window + 1.0)

    if method == "close-to-close":
        ewma_variance = returns.pow(2).ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        return np.sqrt(ewma_variance * 252)

    if method == "parkinson":
        ewma_variance = parkinson_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "garman-klass":
        ewma_variance = garman_klass_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "rogers-satchell":
        ewma_variance = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean().clip(lower=0)
        return np.sqrt(ewma_variance * 252)

    if method == "yang-zhang":
        if window < 2:
            return pd.Series(np.nan, index=ohlc_frame.index)

        k = 0.34 / (1.34 + ((window + 1) / (window - 1)))
        overnight_variance = log_oc.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        open_to_close_variance = log_co.ewm(alpha=alpha, adjust=False, min_periods=window).var(bias=False)
        rs_component = rs_variance.ewm(alpha=alpha, adjust=False, min_periods=window).mean()
        yz_variance = (
            overnight_variance
            + (k * open_to_close_variance)
            + ((1 - k) * rs_component)
        ).clip(lower=0)
        return np.sqrt(yz_variance * 252)

    raise ValueError(f"Unsupported EWMA volatility method: {method}")

volatility_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
default_vol_term = 'long' if 'long' in volatility_term_order else max(
    volatility_term_order,
    key=lambda term: int(time_frame_map[term]),
)

volatility_term_plot_data = {}
for term in volatility_term_order:
    window = int(time_frame_map[term])
    rolling_realized_vol_map = {
        label: compute_rolling_realized_vol_series(window, method)
        for label, method, _, _ in rolling_realized_vol_specs
    }
    ewma_realized_vol_map = {
        label: compute_ewma_realized_vol_series(window, method)
        for label, method, _, _ in ewma_realized_vol_specs
    }

    non_empty_series = [
        series
        for series in (
            [series.dropna() for series in rolling_realized_vol_map.values()]
            + [series.dropna() for series in ewma_realized_vol_map.values()]
            + [model_series.dropna() for model_series in annualized_model_vols.values()]
        )
        if not series.empty
    ]
    if non_empty_series:
        max_index = max(series.index.max() for series in non_empty_series)
        min_index = min(series.index.min() for series in non_empty_series)
        term_range = [max(min_index, max_index - pd.DateOffset(years=3)), max_index]
    else:
        term_range = None

    volatility_term_plot_data[term] = {
        'window': window,
        'rolling_realized_vol_map': rolling_realized_vol_map,
        'ewma_realized_vol_map': ewma_realized_vol_map,
        'term_range': term_range,
    }

vol_model_fig = plot_volatility_model_comparison_view(
    annualized_model_vols=annualized_model_vols,
    term_plot_data=volatility_term_plot_data,
    volatility_model_specs=volatility_model_specs,
    rolling_realized_vol_specs=rolling_realized_vol_specs,
    ewma_realized_vol_specs=ewma_realized_vol_specs,
    term_order=volatility_term_order,
    default_term=default_vol_term,
    time_frame_map=time_frame_map,
    ticker_label=ticker_str,
)
show_plotly_figure(vol_model_fig)